# GéoMarketing IDF : J13b — Enrichir les profils de tous les IRIS d’Île-de-France

## Objectifs
- Conserver tous les IRIS de la base régionale J13a et sa sélection J12.
- Ajouter activité des résidents, catégories professionnelles, ménages et couples.
- Ajouter scolarisation, diplômes, logements et possession d’une voiture.
- Décrire le lieu de travail et les modes de déplacement des résidents.
- Préparer les revenus disponibles, en contrôlant leur géographie avant jointure.
- Calculer des indicateurs lisibles avec leurs vrais dénominateurs.
- Exporter les profils, le dictionnaire, les diagnostics et les cartes séparées.
  
## Périmètre
Tous les IRIS IDF de J13a, y compris les communes non découpées assimilées à un IRIS.
Le focus local est un extrait : aucune commune hors sélection J12 n’est supprimée.
Scénario RP 2022 en géographie 2024. Filosofi 2021 est en géographie 2022.
Un millésime statistique et un millésime géographique sont deux informations distinctes.

## Interprétation
Les variables décrivent les résidents. Les travailleurs résidents ne sont pas les
emplois présents dans le quartier. Les personnes scolarisées de 18–24 ans ne sont
pas nécessairement des étudiants de l’enseignement supérieur, ni les usagers d’un
campus situé dans cet IRIS. Les flux entre communes seront traités au J13d.
Les pourcentages décrivent une structure, à examiner avec les effectifs. Ce notebook
ne prédit ni les ventes ni la rentabilité et ne crée pas de score d’implantation.

In [1]:
#Import des librairies

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from zipfile import ZipFile
from html import escape
import csv
import io
import json
import re
import hashlib
import unicodedata

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 30)
print('Pandas :', pd.__version__, 'GeoPandas :', gpd.__version__)


Pandas : 2.2.2 GeoPandas : 1.0.1


In [2]:
# Chemins du projet

RACINE = Path(os.environ.get('GEOMARKETING_RACINE',
              r'C:\Users\almou\OneDrive\GeoMarketing_IDF'))
FICHIER_BASE = RACINE / 'data/processed/j13a/profil_iris_idf_j13a.csv'
FICHIER_GEO = RACINE / 'data/processed/j13a/profil_iris_idf_j13a.gpkg'
COUCHE_GEO = 'profils_iris'

DOSSIER_RP = RACINE / 'data/raw/insee/iris/rp2022'
FICHIER_ACTIVITE = DOSSIER_RP / 'base-ic-activite-residents-2022_csv.zip'
FICHIER_FORMATION = DOSSIER_RP / 'base-ic-diplomes-formation-2022_csv.zip'
FICHIER_MENAGES = DOSSIER_RP / 'base-ic-couples-familles-menages-2022_csv.zip'
FICHIER_LOGEMENT = DOSSIER_RP / 'base-ic-logement-2022_csv.zip'
FICHIER_REVENUS = (RACINE / 'data/raw/insee/iris/filosofi2021'
                  / 'BASE_TD_FILO_IRIS_2021_DISP_CSV.zip')

DOSSIER_REFERENCES = RACINE / 'data/raw/insee/iris/references'

DOSSIER_INTERIM = RACINE / 'data/interim/j13b'
DOSSIER_PROCESSED = RACINE / 'data/processed/j13b'
DOSSIER_CARTES = RACINE / 'outputs/cartes/j13b'
for dossier in [DOSSIER_INTERIM, DOSSIER_PROCESSED, DOSSIER_CARTES]:
    dossier.mkdir(parents=True, exist_ok=True)




In [3]:
print('Racine :',RACINE)
for nom,fichier in [('J13a CSV',FICHIER_BASE),('J13a contours',FICHIER_GEO),
    ('Activité',FICHIER_ACTIVITE),('Formation',FICHIER_FORMATION),
    ('Ménages',FICHIER_MENAGES),('Logement',FICHIER_LOGEMENT)]:
    print(nom,':',fichier,'— présent :',fichier.is_file())
print('Fichiers disponibles dans',DOSSIER_RP)
if DOSSIER_RP.is_dir():
    for fichier in sorted(DOSSIER_RP.iterdir()):
        if fichier.is_file() and fichier.suffix.lower() in {'.csv','.xlsx','.zip'}:
            print(' -',fichier.name)

Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
J13a CSV : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13a\profil_iris_idf_j13a.csv — présent : True
J13a contours : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13a\profil_iris_idf_j13a.gpkg — présent : True
Activité : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\iris\rp2022\base-ic-activite-residents-2022_csv.zip — présent : True
Formation : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\iris\rp2022\base-ic-diplomes-formation-2022_csv.zip — présent : True
Ménages : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\iris\rp2022\base-ic-couples-familles-menages-2022_csv.zip — présent : True
Logement : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\iris\rp2022\base-ic-logement-2022_csv.zip — présent : True
Fichiers disponibles dans C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\iris\rp2022
 - base-ic-activite-residents-2022_csv.zip
 - base-ic-couples-familles-menages-202

In [4]:
#Paramètres et périmètre

ANNEE_RP = 2022
GEOGRAPHIE_RP = 2024
ANNEE_FILOSOFI = 2021
GEOGRAPHIE_FILOSOFI = 2022
DEPARTEMENTS_IDF = {'75', '77', '78', '91', '92', '93', '94', '95'}
CODES_FOCUS = []  # Exemple : ['95127', '95500'] ; ne réduit pas la base IDF.
CHARGER_REVENUS = True
MODE_COLONNES_MANQUANTES = 'CONTINUER'  # CONTINUER ou ARRETER
SCHEMA_PCS_FORCE = None  # STAT_GSEC, STAT_GSEC_AVEC_SEPARATEUR ou CS, si ambiguïté.
FICHIER_CORRESPONDANCE_REVENUS = None  # Un CSV validé prend priorité sur la méthode automatique.
MODE_JOINTURE_REVENUS = 'REFERENCES_INSEE'  # Ou VALIDATION_CONTOURS pour exiger une preuve des contours.
TELECHARGER_REFERENCES_ABSENTES = True  # Deux ZIP Insee sont conservés intacts dans raw.
FICHIER_REFERENCE_IRIS_REVENUS = None  # Sinon Path(...) vers la référence IRIS 2022.
FICHIER_REFERENCE_IRIS_PROFIL = None  # Sinon Path(...) vers la référence IRIS 2024.

# À préciser seulement si une archive/feuille contient plusieurs tables compatibles.
MEMBRES = {t: None for t in ['ACTIVITE', 'FORMATION', 'MENAGES', 'LOGEMENT', 'REVENUS']}
FEUILLES = MEMBRES.copy()
AFFICHER_CARTES_DANS_NOTEBOOK = False
SIMPLIFICATION_AFFICHAGE_METRES = 10
TOLERANCE_ABSOLUE = 2.0
TOLERANCE_RELATIVE = 0.001
CONFIG_PLOTLY = {'scrollZoom': True, 'displaylogo': False}
P = f'P{str(ANNEE_RP)[-2:]}_'
C = f'C{str(ANNEE_RP)[-2:]}_'
YY_FILO = str(ANNEE_FILOSOFI)[-2:]
ALIAS_IRIS = {'IRIS', 'CODE_IRIS', 'DCOMIRIS'}
ALIAS_COM = {'COM', 'DEPCOM', 'CODGEO', 'CODE_COMMUNE', 'INSEE_COM'}
if min(TOLERANCE_ABSOLUE, TOLERANCE_RELATIVE, SIMPLIFICATION_AFFICHAGE_METRES) < 0:
    raise ValueError('Les tolérances et la simplification doivent être positives ou nulles.')

MODE_COLONNES_MANQUANTES = str(MODE_COLONNES_MANQUANTES).strip().upper()
if MODE_COLONNES_MANQUANTES not in {'CONTINUER','ARRETER'}:
    raise ValueError('MODE_COLONNES_MANQUANTES doit être CONTINUER ou ARRETER.')
if SCHEMA_PCS_FORCE not in {None,'STAT_GSEC','STAT_GSEC_AVEC_SEPARATEUR','CS'}:
    raise ValueError('SCHEMA_PCS_FORCE doit être None, STAT_GSEC, STAT_GSEC_AVEC_SEPARATEUR ou CS.')

if MODE_JOINTURE_REVENUS not in {'REFERENCES_INSEE','VALIDATION_CONTOURS'}:
    raise ValueError('MODE_JOINTURE_REVENUS doit être REFERENCES_INSEE ou VALIDATION_CONTOURS.')



In [5]:
#Fonctions de lecture

LECTURES = []

def normaliser_nom(nom):
    texte = unicodedata.normalize('NFKD', str(nom).strip().upper())
    texte = ''.join(c for c in texte if not unicodedata.combining(c))
    return re.sub(r'[^A-Z0-9]+', '_', texte).strip('_')

def entetes(table):
    geom = table.geometry.name if isinstance(table, gpd.GeoDataFrame) else None
    noms = [c if c == geom else normaliser_nom(c) for c in table.columns]
    if len(noms) != len(set(noms)):
        raise ValueError('Des en-têtes deviennent identiques après normalisation.')
    table = table.copy()
    table.columns = noms
    return table

def normaliser_code(serie, longueur):
    codes = serie.astype('string').str.strip().str.upper().str.replace(r'\.0$', '', regex=True)
    numeriques = codes.str.fullmatch(r'\d{1,' + str(longueur) + '}', na=False)
    codes.loc[numeriques] = codes.loc[numeriques].str.zfill(longueur)
    return codes

def verifier_cle(table, colonne, longueur, nom):
    table = table.copy()
    table[colonne] = normaliser_code(table[colonne], longueur)
    motif = r'(?:\d{2}|2[AB])\d{' + str(longueur - 2) + '}'
    if not table[colonne].str.fullmatch(motif, na=False).all():
        raise ValueError(f'{nom} : codes {colonne} absents ou invalides.')
    if not table[colonne].is_unique:
        doublons = table.loc[table[colonne].duplicated(False), [colonne]]
        exporter_csv(doublons, DOSSIER_INTERIM / ('doublons_' + normaliser_nom(nom) + '.csv'))
        raise ValueError(f'{nom} : plusieurs lignes pour la même clé.')
    return table

def choisir_colonne(table, alias, nom):
    candidats = [c for c in table.columns if c in alias]
    if len(candidats) != 1:
        raise ValueError(f'{nom} : choisir une colonne sans ambiguïté parmi {candidats}.')
    return candidats[0]

def exporter_csv(table, fichier):
    table.to_csv(fichier, index=False, encoding='utf-8-sig')

def sha256(fichier):
    h = hashlib.sha256()
    with Path(fichier).open('rb') as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b''):
            h.update(bloc)
    return h.hexdigest()

@contextmanager
def ouvrir_binaire(fichier, membre=None):
    if membre is None:
        with Path(fichier).open('rb') as flux:
            yield flux
    else:
        with ZipFile(fichier) as archive:
            with archive.open(membre) as flux:
                yield flux

def detecter_table(fichier, groupes, membre=None, feuille=None):
    fichier = Path(fichier)
    if fichier.suffix.lower() == '.zip':
        with ZipFile(fichier) as archive:
            sources = [i.filename for i in archive.infolist()
                       if not i.is_dir() and Path(i.filename).suffix.lower() in {'.csv', '.xlsx'}
                       and not Path(i.filename).name.startswith('~$')]
        if membre is not None:
            if membre not in sources:
                raise ValueError(f'Membre absent de {fichier.name} : {membre}')
            sources = [membre]
    else:
        sources = [None]
    trouves = []
    for source in sources:
        suffixe = Path(source or fichier).suffix.lower()
        if suffixe == '.csv':
            with ouvrir_binaire(fichier, source) as flux:
                lignes_binaires = [flux.readline() for _ in range(30)]
            sample = b''.join(lignes_binaires)
            try:
                sample.decode('utf-8-sig')
                encodage = 'utf-8-sig'
            except UnicodeDecodeError:
                encodage = 'cp1252'
            lignes = sample.decode(encodage).splitlines()
            for sep in [';', ',', '\t']:
                for index, ligne in enumerate(lignes):
                    colonnes = [normaliser_nom(c) for c in next(csv.reader([ligne], delimiter=sep))]
                    if all(set(colonnes) & set(g) for g in groupes):
                        trouves.append({'membre': source, 'feuille': None, 'header': index,
                                         'sep': sep, 'encoding': encodage, 'format': 'csv',
                                         'colonnes_disponibles': colonnes})
        elif suffixe == '.xlsx':
            with ouvrir_binaire(fichier, source) as flux:
                contenu = io.BytesIO(flux.read())
            with pd.ExcelFile(contenu, engine='openpyxl') as classeur:
                feuilles = classeur.sheet_names if feuille is None else [feuille]
                for onglet in feuilles:
                    apercu = pd.read_excel(classeur, sheet_name=onglet, header=None,
                                           nrows=30, dtype='string')
                    for index, ligne in apercu.iterrows():
                        colonnes = [normaliser_nom(c) for c in ligne.dropna().tolist()]
                        if all(set(colonnes) & set(g) for g in groupes):
                            trouves.append({'membre': source, 'feuille': onglet, 'header': int(index),
                                             'sep': None, 'encoding': None, 'format': 'xlsx',
                                             'colonnes_disponibles': colonnes})
        else:
            raise ValueError('Entrée tabulaire attendue : CSV, XLSX ou ZIP contenant ces formats.')
    if len(trouves) > 1:
        csvs = [t for t in trouves if t['format'] == 'csv']
        if len(csvs) == 1 and all(Path(t['membre'] or fichier).stem
                                 == Path(csvs[0]['membre'] or fichier).stem for t in trouves):
            trouves = csvs
    if len(trouves) != 1:
        raise ValueError(f'{fichier.name} : {len(trouves)} tables compatibles. '
                         f'Vérifier les en-têtes ou préciser MEMBRE/FEUILLE. Détections : {trouves}')
    return trouves[0]

def lire_table(fichier, groupes, colonnes, membre=None, feuille=None, info=None):
    if info is None:
        info = detecter_table(fichier, groupes, membre, feuille)
    with ouvrir_binaire(fichier, info['membre']) as flux:
        if info['format'] == 'csv':
            table = pd.read_csv(flux, sep=info['sep'], skiprows=info['header'], header=0,
                encoding=info['encoding'], dtype='string', low_memory=False,
                usecols=lambda c: normaliser_nom(c) in colonnes)
        else:
            table = pd.read_excel(io.BytesIO(flux.read()), sheet_name=info['feuille'],
                header=info['header'], dtype='string', engine='openpyxl',
                usecols=lambda c: normaliser_nom(c) in colonnes)
    table = entetes(table)
    LECTURES.append({'fichier': str(fichier), **info, 'colonnes_lues': table.columns.tolist()})
    print('Lecture :', Path(fichier).name,
          {k:v for k,v in info.items() if k != 'colonnes_disponibles'})
    return table

def nombre(serie):
    texte = serie.astype('string').str.strip()
    texte = texte.str.replace(r'[\s\u00a0\u202f]', '', regex=True).str.replace(',', '.', regex=False)
    return pd.to_numeric(texte, errors='coerce').replace([np.inf, -np.inf], np.nan).astype('Float64')

def renseigne(serie):
    return (serie.astype('string').str.strip().str.upper().notna()
            & ~serie.astype('string').str.strip().str.upper().isin(
                ['', 'NAN', 'NONE', '<NA>', 'NULL', 'ND', '[ND]', '_Z', '_U', 'N/A', '-', '.']))

def verifier_commune_source(table, colonne, nom):
    com = normaliser_code(table[colonne], 5)
    prefixe = table['IRIS'].str[:5]
    parents_plm = {**{f'751{i:02d}': '75056' for i in range(1, 21)},
                   **{f'6938{i}': '69123' for i in range(1, 10)},
                   **{f'132{i:02d}': '13055' for i in range(1, 17)}}
    coherent = com.eq(prefixe) | com.eq(prefixe.map(parents_plm))
    if not coherent.fillna(False).all():
        raise ValueError(f'{nom} : incohérence entre le code IRIS et sa commune/arrondissement.')
    return com


In [6]:
# Marques d’absence/non-diffusion reconnues ; tout autre texte numérique est diagnostiqué.

# Marques d’absence/non-diffusion reconnues ; tout autre texte numérique est diagnostiqué.
MARQUES_ABSENCE = {'', 'NAN', 'NONE', '<NA>', 'NULL', 'ND', '[ND]', 'NS', 'S', 'SS',
                   '_S', '_Z', '_U', '_N', '_M', 'N/A', '-', '.', '...', '///', '*', '**', '***'}
def renseigne(serie):
    texte = serie.astype('string').str.strip().str.upper()
    return texte.notna() & ~texte.isin(MARQUES_ABSENCE)

def booleen(serie, nom):
    texte = serie.astype('string').str.strip().str.upper()
    valeurs = texte.map({'TRUE': True, 'FALSE': False, '1': True, '0': False,
                         'VRAI': True, 'FAUX': False})
    if valeurs.isna().any():
        raise ValueError(f'{nom} : valeur booléenne absente ou non reconnue.')
    return valeurs.astype(bool)

CONTROLES_JOINTURES = []
QUALITE_VARIABLES = []
CONTROLES_EFFECTIFS = []
DICTIONNAIRE = []
RATIOS = {}

DIAGNOSTIC_COLONNES = []
MANIFESTE_SOURCES = []
COLONNES_PRESENTES = {}
STATUT_PCS_SOURCE = 'NON_DISPONIBLE'


def enregistrer_indicateur(description):
    # Une réexécution met à jour la description de cette colonne, à la même position.
    colonne = description.get('COLONNE')
    if not isinstance(colonne,str) or not colonne.strip():
        raise ValueError('Description d’indicateur : nom COLONNE absent ou invalide.')
    positions = [i for i,ligne in enumerate(DICTIONNAIRE) if ligne.get('COLONNE')==colonne]
    if positions:
        DICTIONNAIRE[positions[0]] = dict(description)
        for i in reversed(positions[1:]):
            del DICTIONNAIRE[i]
    else:
        DICTIONNAIRE.append(dict(description))

def valider_dictionnaire():
    if not DICTIONNAIRE:
        raise ValueError('Dictionnaire vide : exécuter les blocs de calcul avant les exports.')
    table = pd.DataFrame(DICTIONNAIRE)
    if 'COLONNE' not in table or table['COLONNE'].isna().any() or table['COLONNE'].astype('string').str.strip().eq('').any():
        raise ValueError('Dictionnaire : une description ne possède pas de nom de colonne.')
    # Des copies entièrement identiques peuvent provenir d’une ancienne cellule réexécutée.
    garder = ~table.duplicated()
    uniques = [ligne for ligne,retenir in zip(DICTIONNAIRE,garder) if retenir]
    table_unique = pd.DataFrame(uniques)
    conflits = table_unique.loc[table_unique['COLONNE'].duplicated(keep=False)]
    if not conflits.empty:
        exporter_csv(conflits,DOSSIER_INTERIM/'conflits_dictionnaire_j13b.csv')
        noms = sorted(conflits['COLONNE'].unique())
        raise ValueError('Dictionnaire : descriptions différentes pour les mêmes colonnes : '
                         +str(noms)+'. Consulter conflits_dictionnaire_j13b.csv.')
    DICTIONNAIRE[:] = uniques
    return table_unique



In [7]:
for fichier in [FICHIER_BASE, FICHIER_GEO, FICHIER_ACTIVITE,
                 FICHIER_FORMATION, FICHIER_MENAGES, FICHIER_LOGEMENT]:
    if not fichier.is_file():
        raise FileNotFoundError(f'Fichier absent : {fichier}. Exécuter J13a ou adapter les chemins.')
if CHARGER_REVENUS and not FICHIER_REVENUS.is_file():
    raise FileNotFoundError(f'Télécharger la base des revenus disponibles : {FICHIER_REVENUS}. '
                            'CHARGER_REVENUS=False permet de traiter seulement les thèmes RP.')
base = entetes(pd.read_csv(FICHIER_BASE, sep=None, engine='python',
                          dtype='string', encoding='utf-8-sig'))
base = verifier_cle(base, 'IRIS', 9, 'Base J13a')
requis_base = {'CODGEO_SOURCE', 'CODGEO_ETUDE', 'NOM_IRIS', 'NOM_COMMUNE_ETUDE',
    'J13A_ANNEE_RP', 'J13A_GEOGRAPHIE', 'J13A_POPULATION', 'J13A_SURFACE_KM2',
    'J13A_COMMUNE_PRESELECTIONNEE_J12', 'J13A_DANS_FOCUS'}
if requis_base - set(base.columns):
    raise ValueError(f'Colonnes J13a absentes : {sorted(requis_base-set(base.columns))}')
if any(c.startswith('J13B_') for c in base.columns):
    raise ValueError('Le fichier d’entrée doit être J13a, pas un ancien export J13b.')
base['CODGEO_SOURCE'] = normaliser_code(base['CODGEO_SOURCE'], 5)
base['CODGEO_ETUDE'] = normaliser_code(base['CODGEO_ETUDE'], 5)
if not base['CODGEO_SOURCE'].eq(base['IRIS'].str[:5]).all():
    raise ValueError('CODGEO_SOURCE ne correspond pas au code IRIS.')
base['DEPARTEMENT'] = base['IRIS'].str[:2]
if set(base['DEPARTEMENT']) != DEPARTEMENTS_IDF:
    raise ValueError('La base ne couvre pas les huit départements IDF, ou contient un autre département.')
annees_base = nombre(base['J13A_ANNEE_RP']); geographies_base = nombre(base['J13A_GEOGRAPHIE'])
if (not annees_base.notna().all() or not geographies_base.notna().all()
        or not annees_base.eq(ANNEE_RP).all() or not geographies_base.eq(GEOGRAPHIE_RP).all()):
    raise ValueError('Millésime/géographie J13a incompatibles avec les paramètres J13b.')
if not base['CODGEO_ETUDE'].str.fullmatch(r'\d{5}',na=False).all():
    raise ValueError('Codes des communes de l’étude invalides dans J13a.')
for col in ['J13A_COMMUNE_PRESELECTIONNEE_J12', 'J13A_DANS_FOCUS']:
    base[col] = booleen(base[col], col)
if CODES_FOCUS:
    codes_focus = set(normaliser_code(pd.Series(CODES_FOCUS), 5))
    absents = codes_focus-set(base['CODGEO_ETUDE'])
    if absents:
        raise ValueError(f'Communes du focus absentes de la base IDF : {sorted(absents)}')
    base['J13B_DANS_FOCUS'] = base['CODGEO_ETUDE'].isin(codes_focus)
else:
    base['J13B_DANS_FOCUS'] = base['J13A_DANS_FOCUS']
    codes_focus = set(base.loc[base['J13B_DANS_FOCUS'], 'CODGEO_ETUDE'])
ATTENDUS = set(base['IRIS'])
NOMBRE_IRIS_INITIAL = len(base)
profil = base.copy()
print('IRIS IDF :', len(base), '· communes :', base['CODGEO_ETUDE'].nunique(),
      '· IRIS du focus :', int(base['J13B_DANS_FOCUS'].sum()))
display(base[['IRIS', 'NOM_IRIS', 'NOM_COMMUNE_ETUDE', 'J13B_DANS_FOCUS']].head(10))


IRIS IDF : 5264 · communes : 1266 · IRIS du focus : 168


,IRIS,NOM_IRIS,NOM_COMMUNE_ETUDE,J13B_DANS_FOCUS
0,751031101,Les Archives 1,Paris,False
1,751051702,Saint-Victor 2,Paris,False
2,751062309,Notre-Dame des Champs 9,Paris,False
3,751072503,Saint-Thomas d'Aquin 3,Paris,False
4,751135029,Gare 29,Paris,False
5,751135104,Maison Blanche 4,Paris,False
6,751135206,Croulebarbe 6,Paris,False
7,751145512,Petit Montrouge 12,Paris,False
8,751187008,Clignancourt 8,Paris,False
9,751187017,Clignancourt 17,Paris,False


In [8]:
info_activite = detecter_table(FICHIER_ACTIVITE,[ALIAS_IRIS,ALIAS_COM],
                               MEMBRES['ACTIVITE'],FEUILLES['ACTIVITE'])
print('Fichier :',FICHIER_ACTIVITE)
print('Membre ZIP :',info_activite['membre'],'; feuille :',info_activite['feuille'])
print('Nombre de colonnes :',len(info_activite['colonnes_disponibles']))
for colonne in info_activite['colonnes_disponibles']:
    print(colonne)
exporter_csv(pd.DataFrame({'COLONNE':info_activite['colonnes_disponibles']}),
             DOSSIER_INTERIM/'entetes_activite_j13b.csv')

Fichier : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\iris\rp2022\base-ic-activite-residents-2022_csv.zip
Membre ZIP : base-ic-activite-residents-2022.CSV ; feuille : None
Nombre de colonnes : 119
IRIS
COM
TYP_IRIS
LAB_IRIS
P22_POP1564
P22_POP1524
P22_POP2554
P22_POP5564
P22_H1564
P22_H1524
P22_H2554
P22_H5564
P22_F1564
P22_F1524
P22_F2554
P22_F5564
P22_ACT1564
P22_ACT1524
P22_ACT2554
P22_ACT5564
P22_HACT1564
P22_HACT1524
P22_HACT2554
P22_HACT5564
P22_FACT1564
P22_FACT1524
P22_FACT2554
P22_FACT5564
P22_ACTOCC1564
P22_ACTOCC1524
P22_ACTOCC2554
P22_ACTOCC5564
P22_HACTOCC1564
P22_HACTOCC1524
P22_HACTOCC2554
P22_HACTOCC5564
P22_FACTOCC1564
P22_FACTOCC1524
P22_FACTOCC2554
P22_CHOM_CAPBEP
P22_FACTOCC5564
P22_CHOM_BAC
P22_CHOM1564
P22_CHOM_SUP2
P22_CHOM1524
P22_CHOM_SUP34
P22_CHOM2554
P22_CHOM_SUP5
P22_CHOM5564
P22_ACT_DIPLMIN
P22_CHOM_DIPLMIN
P22_ACT_BEPC
P22_CHOM_BEPC
P22_ACT_CAPBEP
P22_ACT_BAC
P22_ACT_SUP2
P22_ACT_SUP34
P22_ACT_SUP5
P22_INACT1564
P22_HINACT1564
P22_FINACT1564
P

In [9]:
#Définir les variables de chaque thème

def specifications(prefixe, lignes):
    return {prefixe+s: ('J13B_'+d, lib) for s, d, lib in lignes}

VAR_ACT = specifications(P, [
 ('POP1564','POP_15_64','Population de 15 à 64 ans'),
 ('ACT1564','NB_ACTIFS_15_64','Actifs résidents de 15 à 64 ans'),
 ('ACTOCC1564','NB_ACTIFS_OCCUPES_15_64_P','Actifs occupés résidents de 15 à 64 ans — principale'),
 ('CHOM1564','NB_CHOMEURS_15_64','Chômeurs au sens du recensement de 15 à 64 ans'),
 ('INACT1564','NB_INACTIFS_15_64','Inactifs résidents de 15 à 64 ans'),
 ('ETUD1564','NB_ELEVES_ETUDIANTS_STAGIAIRES_15_64','Élèves, étudiants et stagiaires non rémunérés de 15 à 64 ans'),
 ('ACTOCC15P','NB_ACTIFS_OCCUPES_15P_P','Actifs occupés résidents de 15 ans ou plus — principale'),
 ('ACTOCC15P_ILT1','NB_ACTIFS_TRAVAIL_COMMUNE_RESIDENCE','Actifs occupés travaillant dans leur commune de résidence'),
 ('ACTOCC15P_ILT2P','NB_ACTIFS_TRAVAIL_AUTRE_COMMUNE','Actifs occupés travaillant dans une autre commune'),
])
VAR_ACT.update(specifications(C, [
 ('ACTOCC1564','NB_ACTIFS_OCCUPES_15_64_C','Actifs occupés résidents de 15 à 64 ans — complémentaire'),
 ('ACTOCC15P','NB_ACTIFS_OCCUPES_15P_C','Actifs occupés résidents de 15 ans ou plus — complémentaire'),
 ('ACTOCC15P_PAS','NB_ACTIFS_SANS_TRANSPORT','Actifs occupés sans transport pour aller travailler'),
 ('ACTOCC15P_MAR','NB_ACTIFS_MARCHE','Actifs occupés allant principalement travailler à pied'),
 ('ACTOCC15P_VELO','NB_ACTIFS_VELO','Actifs occupés utilisant principalement le vélo'),
 ('ACTOCC15P_2ROUESMOT','NB_ACTIFS_DEUX_ROUES_MOTORISES','Actifs occupés utilisant principalement un deux-roues motorisé'),
 ('ACTOCC15P_VOIT','NB_ACTIFS_VOITURE','Actifs occupés utilisant principalement une voiture ou un camion'),
 ('ACTOCC15P_TCOM','NB_ACTIFS_TRANSPORTS_COMMUN','Actifs occupés utilisant principalement les transports en commun'),
]))
PCS = {'C22_ACTOCC1564_STAT_GSEC11':('AGRICULTEURS','Agriculteurs exploitants'),
       'C22_ACTOCC1564_STAT_GSEC12':('ARTISANS_COMMERCANTS_CHEFS_ENTREPRISE','Artisans, commerçants et chefs d’entreprise'),
       'C22_ACTOCC1564_STAT_GSEC13':('CADRES','Cadres et professions intellectuelles supérieures'),
       'C22_ACTOCC1564_STAT_GSEC14':('PROFESSIONS_INTERMEDIAIRES','Professions intermédiaires'),
       'C22_ACTOCC1564_STAT_GSEC15':('EMPLOYES','Employés'),
       'C22_ACTOCC1564_STAT_GSEC16':('OUVRIERS','Ouvriers')}

# Les clés de PCS sont les noms sources complets fournis pour le fichier RP 2022.
# GSEC11, GSEC12... ne comportent pas de séparateur entre GSEC et le numéro.
if any(not source.startswith(C+'ACTOCC1564_') for source in PCS):
    raise ValueError('La correspondance PCS fournie concerne RP 2022. '
                     'Adapter ses noms sources avant de changer ANNEE_RP.')
SCHEMAS_PCS = {'STAT_GSEC': {}, 'STAT_GSEC_AVEC_SEPARATEUR': {}, 'CS': {}}
for source, (suffixe, libelle) in PCS.items():
    numero = source[-2:]
    destination = ('J13B_NB_ACTIFS_OCCUPES_'+suffixe+'_15_64',
                   libelle+' en emploi, résidents de 15 à 64 ans')
    SCHEMAS_PCS['STAT_GSEC'][source] = destination
    SCHEMAS_PCS['STAT_GSEC_AVEC_SEPARATEUR'][C+'ACTOCC1564_STAT_GSEC_'+numero] = destination
    SCHEMAS_PCS['CS'][C+'ACTOCC1564_CS'+str(int(numero)-10)] = destination
VAR_ACT.update(SCHEMAS_PCS['STAT_GSEC'])
SCHEMA_PCS_CHOISI = None

def resoudre_pcs(spec, colonnes_disponibles):
    disponibles = set(colonnes_disponibles)
    completes = [nom for nom,serie in SCHEMAS_PCS.items() if set(serie).issubset(disponibles)]
    observees = [nom for nom,serie in SCHEMAS_PCS.items() if set(serie)&disponibles]
    if SCHEMA_PCS_FORCE is not None:
        nom = SCHEMA_PCS_FORCE
    elif len(completes)==1:
        nom = completes[0]
    elif len(completes)>1 or len(observees)>1:
        raise ValueError('ACTIVITE : plusieurs conventions PCS sont présentes. '
            'Examiner entetes_activite_j13b.csv et préciser SCHEMA_PCS_FORCE au bloc 4. '
            'Les catégories de deux séries ne sont pas assemblées automatiquement.')
    elif observees:
        nom = observees[0]
    else:
        nom = None
    toutes_pcs = {col for serie in SCHEMAS_PCS.values() for col in serie}
    adaptee = {col:definition for col,definition in spec.items() if col not in toutes_pcs}
    # Conserver six colonnes de sortie stables, même quand aucune série n’est fournie.
    attendue = SCHEMAS_PCS[nom or 'STAT_GSEC']
    adaptee.update(attendue)
    nb = len(set(attendue)&disponibles)
    statut = 'COMPLETE' if nb==6 else ('PARTIELLE' if nb else 'NON_DISPONIBLE')
    return adaptee, nom, statut

AGES_SCOL = {'0205':'2–5 ans','0610':'6–10 ans','1114':'11–14 ans',
             '1517':'15–17 ans','1824':'18–24 ans','2529':'25–29 ans','30P':'30 ans ou plus'}
DIPLOMES = {'DIPLMIN':('SANS_DIPLOME_CEP','Sans diplôme ou au plus un CEP'),
 'BEPC':('BEPC_BREVET','BEPC, brevet des collèges ou DNB'),
 'CAPBEP':('CAP_BEP','CAP, BEP ou équivalent'), 'BAC':('BAC','Baccalauréat ou équivalent'),
 'SUP2':('BAC_PLUS_2','Bac + 2'), 'SUP34':('BAC_PLUS_3_4','Bac + 3 ou Bac + 4'),
 'SUP5':('BAC_PLUS_5_OU_PLUS','Bac + 5 ou plus')}
VAR_FORM = {P+'NSCOL15P':('J13B_NB_NON_SCOLARISES_15P','Personnes non scolarisées de 15 ans ou plus')}
for age, libelle in AGES_SCOL.items():
    VAR_FORM[P+'POP'+age] = ('J13B_POP_'+age,'Population de '+libelle)
    VAR_FORM[P+'SCOL'+age] = ('J13B_NB_SCOLARISES_'+age,'Personnes scolarisées de '+libelle)
for source, (suffixe, libelle) in DIPLOMES.items():
    VAR_FORM[P+'NSCOL15P_'+source] = ('J13B_NB_NON_SCOLARISES_15P_'+suffixe,libelle+' parmi les non-scolarisés de 15 ans ou plus')

VAR_MEN = specifications(C, [
 ('MEN','NB_MENAGES_C','Ménages — complémentaire'),
 ('PMEN','POP_MENAGES_C','Personnes des ménages — complémentaire'),
 ('MENPSEUL','NB_MENAGES_UNE_PERSONNE','Ménages d’une personne'),
 ('MENHSEUL','NB_MENAGES_HOMME_SEUL','Ménages composés d’un homme seul'),
 ('MENFSEUL','NB_MENAGES_FEMME_SEULE','Ménages composés d’une femme seule'),
 ('MENSFAM','NB_MENAGES_AUTRES_SANS_FAMILLE','Autres ménages sans famille'),
 ('MENFAM','NB_MENAGES_AVEC_FAMILLE','Ménages avec famille(s)'),
 ('MENCOUPSENF','NB_MENAGES_COUPLE_SANS_ENFANT','Ménages dont la famille principale est un couple sans enfant'),
 ('MENCOUPAENF','NB_MENAGES_COUPLE_AVEC_ENFANTS','Ménages dont la famille principale est un couple avec enfant(s)'),
 ('MENFAMMONO','NB_MENAGES_FAMILLE_MONOPARENTALE','Ménages dont la famille principale est une famille monoparentale'),
])
VAR_MEN.update(specifications(P, [('POP15P','POP_15P_STATUT_CONJUGAL','Population de 15 ans ou plus — statut conjugal')]))
CONJUGAL = {'MARIEE':('MARIEES','Mariées'),'PACSEE':('PACSEES','Pacsées'),
 'CONCUB_UNION_LIBRE':('CONCUBINAGE','En concubinage ou union libre'),
 'VEUFS':('VEUVES','Veuves ou veufs'),'DIVORCEE':('DIVORCEES','Divorcées'),
 'CELIBATAIRE':('CELIBATAIRES','Célibataires')}
for source, (suffixe, libelle) in CONJUGAL.items():
    VAR_MEN[P+'POP15P_'+source] = ('J13B_NB_PERSONNES_15P_'+suffixe,libelle+' — personnes de 15 ans ou plus')

VAR_LOG = specifications(P, [
 ('LOG','NB_LOGEMENTS','Logements'), ('RP','NB_RESIDENCES_PRINCIPALES','Résidences principales'),
 ('RSECOCC','NB_RESIDENCES_SECONDAIRES_OCCASIONNELLES','Résidences secondaires et logements occasionnels'),
 ('LOGVAC','NB_LOGEMENTS_VACANTS','Logements vacants'),
 ('MAISON','NB_MAISONS','Logements de type maison'), ('APPART','NB_APPARTEMENTS','Logements de type appartement'),
 ('RP_PROP','NB_RP_PROPRIETAIRES','Résidences principales occupées par des propriétaires'),
 ('RP_LOC','NB_RP_LOCATAIRES','Résidences principales occupées par des locataires'),
 ('RP_LOCHLMV','NB_RP_HLM_LOUE_VIDE','Résidences principales HLM louées vides'),
 ('RP_GRAT','NB_RP_OCCUPATION_GRATUITE','Résidences principales occupées gratuitement'),
 ('MEN','NB_MENAGES_P','Ménages — principale'),
 ('RP_VOIT1P','NB_MENAGES_AU_MOINS_UNE_VOITURE','Ménages disposant d’au moins une voiture'),
 ('RP_VOIT1','NB_MENAGES_UNE_VOITURE','Ménages disposant d’une voiture'),
 ('RP_VOIT2P','NB_MENAGES_DEUX_VOITURES_OU_PLUS','Ménages disposant de deux voitures ou plus'),
 ('MEN_ANEM0002','NB_MENAGES_EMMENAGES_MOINS_2_ANS','Ménages ayant emménagé depuis moins de deux ans'),
])
THEMES = {'ACTIVITE':(FICHIER_ACTIVITE,VAR_ACT), 'FORMATION':(FICHIER_FORMATION,VAR_FORM),
          'MENAGES':(FICHIER_MENAGES,VAR_MEN), 'LOGEMENT':(FICHIER_LOGEMENT,VAR_LOG)}
COLONNES_EFFECTIFS = [dest for _, spec in THEMES.values() for dest, _ in spec.values()]
if len(COLONNES_EFFECTIFS) != len(set(COLONNES_EFFECTIFS)):
    raise ValueError('Deux variables sources ont le même nom de sortie.')
print('Variables sélectionnées par thème :', {t:len(v) for t,(_,v) in THEMES.items()})

# Ces repères distinguent les thèmes. POP1564 seul ne suffit pas à identifier Activité.
REPERES_THEMES = {
    'ACTIVITE':{P+'ACT1564',P+'ACTOCC1564',C+'ACTOCC1564',C+'ACTOCC15P'},
    'FORMATION':{P+'NSCOL15P'} | {P+'SCOL'+a for a in AGES_SCOL},
    'MENAGES':{C+'MEN',C+'MENPSEUL',C+'MENFAM',C+'MENHSEUL',C+'MENFSEUL'},
    'LOGEMENT':{P+'LOG',P+'RP',P+'LOGVAC',P+'MAISON',P+'APPART'},
}


Variables sélectionnées par thème : {'ACTIVITE': 23, 'FORMATION': 22, 'MENAGES': 17, 'LOGEMENT': 15}


In [10]:
info_activite = detecter_table(FICHIER_ACTIVITE,[ALIAS_IRIS,ALIAS_COM],
                               MEMBRES['ACTIVITE'],FEUILLES['ACTIVITE'])
print('Fichier :',FICHIER_ACTIVITE)
print('Membre ZIP :',info_activite['membre'],'; feuille :',info_activite['feuille'])
print('Nombre de colonnes :',len(info_activite['colonnes_disponibles']))
for colonne in info_activite['colonnes_disponibles']:
    print(colonne)
exporter_csv(pd.DataFrame({'COLONNE':info_activite['colonnes_disponibles']}),
             DOSSIER_INTERIM/'entetes_activite_j13b.csv')

Fichier : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\iris\rp2022\base-ic-activite-residents-2022_csv.zip
Membre ZIP : base-ic-activite-residents-2022.CSV ; feuille : None
Nombre de colonnes : 119
IRIS
COM
TYP_IRIS
LAB_IRIS
P22_POP1564
P22_POP1524
P22_POP2554
P22_POP5564
P22_H1564
P22_H1524
P22_H2554
P22_H5564
P22_F1564
P22_F1524
P22_F2554
P22_F5564
P22_ACT1564
P22_ACT1524
P22_ACT2554
P22_ACT5564
P22_HACT1564
P22_HACT1524
P22_HACT2554
P22_HACT5564
P22_FACT1564
P22_FACT1524
P22_FACT2554
P22_FACT5564
P22_ACTOCC1564
P22_ACTOCC1524
P22_ACTOCC2554
P22_ACTOCC5564
P22_HACTOCC1564
P22_HACTOCC1524
P22_HACTOCC2554
P22_HACTOCC5564
P22_FACTOCC1564
P22_FACTOCC1524
P22_FACTOCC2554
P22_CHOM_CAPBEP
P22_FACTOCC5564
P22_CHOM_BAC
P22_CHOM1564
P22_CHOM_SUP2
P22_CHOM1524
P22_CHOM_SUP34
P22_CHOM2554
P22_CHOM_SUP5
P22_CHOM5564
P22_ACT_DIPLMIN
P22_CHOM_DIPLMIN
P22_ACT_BEPC
P22_CHOM_BEPC
P22_ACT_CAPBEP
P22_ACT_BAC
P22_ACT_SUP2
P22_ACT_SUP34
P22_ACT_SUP5
P22_INACT1564
P22_HINACT1564
P22_FINACT1564
P

In [11]:
#Controler, inspecter les 4 sources

sources_rp = {}
for theme,(fichier,spec) in THEMES.items():
    # Inspecter toutes les colonnes avant usecols ; conserver le vrai membre et la feuille.
    groupes = [ALIAS_IRIS,ALIAS_COM]
    info = detecter_table(fichier,groupes,MEMBRES[theme],FEUILLES[theme])
    disponibles = set(info['colonnes_disponibles'])
    exporter_csv(pd.DataFrame({'COLONNE':info['colonnes_disponibles']}),
                 DOSSIER_INTERIM/f'entetes_{theme.lower()}_j13b.csv')
    if not disponibles & REPERES_THEMES[theme]:
        raise ValueError(f'{theme} : aucun repère du thème RP {ANNEE_RP} dans les en-têtes. '
            f'Verifier le fichier {fichier.name}, le membre ZIP, la feuille et le millésime. '
            f'Consulter entetes_{theme.lower()}_j13b.csv. Une autre base ne sera pas traitée comme ce thème.')
    if theme=='ACTIVITE':
        spec,SCHEMA_PCS_CHOISI,STATUT_PCS_SOURCE = resoudre_pcs(spec,info['colonnes_disponibles'])
        VAR_ACT = spec
        THEMES[theme] = (fichier,spec)
        print('PCS :',STATUT_PCS_SOURCE,'; convention :',SCHEMA_PCS_CHOISI or 'aucune reconnue')
    absentes = sorted(set(spec)-disponibles)
    for source,(destination,libelle) in spec.items():
        DIAGNOSTIC_COLONNES.append({'THEME':theme,'VARIABLE_SOURCE':source,
            'COLONNE_SORTIE':destination,'LIBELLE':libelle,'PRESENTE':source in disponibles,
            'ETAT':'COLONNE_PRESENTE' if source in disponibles else 'COLONNE_ABSENTE'})
    MANIFESTE_SOURCES.append({'THEME':theme,'FICHIER':str(fichier),
        'MEMBRE_ZIP':info['membre'],'FEUILLE_XLSX':info['feuille'],
        'ANNEE_RP_DECLAREE':ANNEE_RP,'GEOGRAPHIE_DECLAREE':GEOGRAPHIE_RP,
        'NB_COLONNES_DISPONIBLES':len(disponibles),'NB_VARIABLES_ATTENDUES':len(spec),
        'NB_VARIABLES_PRESENTES':len(set(spec)&disponibles),'NB_VARIABLES_ABSENTES':len(absentes),
        'STATUT':'PARTIELLE' if absentes else 'COMPLETE'})
    exporter_csv(pd.DataFrame(DIAGNOSTIC_COLONNES),DOSSIER_INTERIM/'disponibilite_colonnes_source_j13b.csv')
    exporter_csv(pd.DataFrame(MANIFESTE_SOURCES),DOSSIER_INTERIM/'manifest_sources_j13b.csv')
    if absentes:
        print(theme,':',len(absentes),'colonne(s) absente(s). Indicateurs concernés : NA.')
        print('Colonnes absentes :',absentes)
        if MODE_COLONNES_MANQUANTES=='ARRETER':
            raise ValueError(f'{theme} : colonnes absentes en mode ARRETER. Consulter disponibilite_colonnes_source_j13b.csv.')
    table = lire_table(fichier,groupes,ALIAS_IRIS|ALIAS_COM|set(spec),
                       MEMBRES[theme],FEUILLES[theme],info=info)
    ci = choisir_colonne(table,ALIAS_IRIS,theme+' — IRIS')
    cc = choisir_colonne(table,ALIAS_COM,theme+' — commune')
    table = verifier_cle(table.rename(columns={ci:'IRIS'}),'IRIS',9,theme)
    table['COMMUNE_SOURCE_RP'] = verifier_commune_source(table,cc,theme)
    COLONNES_PRESENTES[theme] = set(table.columns)
    table = table[table['IRIS'].str[:2].isin(DEPARTEMENTS_IDF)].copy()
    presents = set(table['IRIS'])
    diagnostic = pd.DataFrame(
        [{'IRIS':c,'ETAT':'LIGNE_SOURCE_ABSENTE'} for c in sorted(ATTENDUS-presents)]
        + [{'IRIS':c,'ETAT':'LIGNE_SOURCE_SUPPLEMENTAIRE'} for c in sorted(presents-ATTENDUS)],
        columns=['IRIS','ETAT'])
    exporter_csv(diagnostic,DOSSIER_INTERIM/f'diagnostic_cles_{theme.lower()}_j13b.csv')
    CONTROLES_JOINTURES.append({'THEME':theme,'NB_IRIS_ATTENDUS':len(ATTENDUS),
        'NB_LIGNES_SOURCE_IDF':len(table),'NB_ABSENTS':len(ATTENDUS-presents),
        'NB_SUPPLEMENTAIRES':len(presents-ATTENDUS),'NB_COLONNES_ABSENTES':len(absentes)})
    if not diagnostic.empty:
        raise ValueError(f'{theme} : couverture différente de J13a. Examiner les codes et la géographie.')
    sources_rp[theme] = table
    # L’extraction source contient seulement les champs réellement lus.
    exporter_csv(table,DOSSIER_INTERIM/f'{theme.lower()}_iris_idf_source_j13b.csv')
exporter_csv(pd.DataFrame(CONTROLES_JOINTURES),DOSSIER_INTERIM/'controle_jointures_rp_j13b.csv')
display(pd.DataFrame(MANIFESTE_SOURCES))
display(pd.DataFrame(CONTROLES_JOINTURES))



PCS : COMPLETE ; convention : STAT_GSEC
Lecture : base-ic-activite-residents-2022_csv.zip {'membre': 'base-ic-activite-residents-2022.CSV', 'feuille': None, 'header': 0, 'sep': ';', 'encoding': 'utf-8-sig', 'format': 'csv'}
Lecture : base-ic-diplomes-formation-2022_csv.zip {'membre': 'base-ic-diplomes-formation-2022.CSV', 'feuille': None, 'header': 0, 'sep': ';', 'encoding': 'utf-8-sig', 'format': 'csv'}
Lecture : base-ic-couples-familles-menages-2022_csv.zip {'membre': 'base-ic-couples-familles-menages-2022.CSV', 'feuille': None, 'header': 0, 'sep': ';', 'encoding': 'utf-8-sig', 'format': 'csv'}
Lecture : base-ic-logement-2022_csv.zip {'membre': 'base-ic-logement-2022.CSV', 'feuille': None, 'header': 0, 'sep': ';', 'encoding': 'utf-8-sig', 'format': 'csv'}


,THEME,FICHIER,MEMBRE_ZIP,FEUILLE_XLSX,ANNEE_RP_DECLAREE,GEOGRAPHIE_DECLAREE,NB_COLONNES_DISPONIBLES,NB_VARIABLES_ATTENDUES,NB_VARIABLES_PRESENTES,NB_VARIABLES_ABSENTES,STATUT
0,ACTIVITE,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,base-ic-activite-residents-2022.CSV,None,2022,2024,119,23,23,0,COMPLETE
1,FORMATION,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,base-ic-diplomes-formation-2022.CSV,None,2022,2024,42,22,22,0,COMPLETE
2,MENAGES,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,base-ic-couples-familles-menages-2022.CSV,None,2022,2024,80,17,17,0,COMPLETE
3,LOGEMENT,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,base-ic-logement-2022.CSV,None,2022,2024,101,15,15,0,COMPLETE


,THEME,NB_IRIS_ATTENDUS,NB_LIGNES_SOURCE_IDF,NB_ABSENTS,NB_SUPPLEMENTAIRES,NB_COLONNES_ABSENTES
0,ACTIVITE,5264,5264,0,0,0
1,FORMATION,5264,5264,0,0,0
2,MENAGES,5264,5264,0,0,0
3,LOGEMENT,5264,5264,0,0,0


In [12]:
#Convertir les effectifs

anomalies_conversion = []
for theme, (_, spec) in THEMES.items():
    selection = sources_rp[theme][['IRIS']].copy()
    for source, (destination, libelle) in spec.items():
        presente = source in COLONNES_PRESENTES[theme]
        raw = (sources_rp[theme][source] if presente else
               pd.Series(pd.NA,index=sources_rp[theme].index,dtype='string'))
        valeurs = nombre(raw).where(renseigne(raw))
        mauvais = (renseigne(raw) & valeurs.isna()) | valeurs.lt(0).fillna(False)
        for index in raw.index[mauvais]:
            anomalies_conversion.append({'THEME':theme,'IRIS':sources_rp[theme].at[index,'IRIS'],
                                          'VARIABLE':source,'VALEUR':str(raw.loc[index])})
        selection[destination] = valeurs
        QUALITE_VARIABLES.append({'THEME':theme,'VARIABLE':source,'COLONNE_SORTIE':destination,
            'COLONNE_SOURCE_PRESENTE':presente,'NB_RENSEIGNES':int(valeurs.notna().sum()),
            'NB_MANQUANTS':int(valeurs.isna().sum())})
        enregistrer_indicateur({'COLONNE':destination,'LIBELLE':libelle,'THEME':theme,
            'TYPE':'EFFECTIF','STATUT_SOURCE':'COLONNE_PRESENTE' if presente else 'COLONNE_ABSENTE',
            'UNITE':'personnes / ménages / logements selon le libellé',
            'NUMERATEUR':source,'DENOMINATEUR':'','FACTEUR':1,'AGREGATION':'SOMME_SI_COMPLETE',
            'ANNEE':ANNEE_RP,'GEOGRAPHIE_SOURCE':GEOGRAPHIE_RP})
    selection['J13B_LIGNE_'+theme+'_PRESENTE'] = True
    nb_absentes = sum(s not in COLONNES_PRESENTES[theme] for s in spec)
    selection['J13B_NB_COLONNES_SOURCE_'+theme+'_ABSENTES'] = nb_absentes
    selection['J13B_STATUT_SOURCE_'+theme] = 'PARTIELLE' if nb_absentes else 'COMPLETE'
    profil = profil.merge(selection, on='IRIS', how='left', validate='one_to_one')
    assert len(profil)==NOMBRE_IRIS_INITIAL
exporter_csv(pd.DataFrame(anomalies_conversion,columns=['THEME','IRIS','VARIABLE','VALEUR']),
              DOSSIER_INTERIM / 'anomalies_conversion_j13b.csv')
if anomalies_conversion:
    raise ValueError('Valeurs sources invalides : consulter anomalies_conversion_j13b.csv.')

def verifier_effectif(numerateur, denominateur, nom, egalite=False):
    n = nombre(numerateur); d = nombre(denominateur)
    connus = n.notna() & d.notna()
    tol = (d.abs()*TOLERANCE_RELATIVE).clip(lower=1e-6,upper=TOLERANCE_ABSOLUE)
    mauvais = ((n-d).abs()>tol) if egalite else ((n>d+tol) | (n<0))
    mauvais = (mauvais | (d.eq(0) & n.ne(0))).fillna(False) & connus
    CONTROLES_EFFECTIFS.append({'CONTROLE':nom,'NB_COMPARABLES':int(connus.sum()),
        'NB_NON_COMPARABLES':int((~connus).sum()),'NB_ANOMALIES':int(mauvais.sum()),
        'ETAT':'NON_VERIFIABLE' if not connus.any() else ('PARTIEL' if not connus.all() else 'COMPLET')})
    if mauvais.any():
        diagnostic = profil.loc[mauvais,['IRIS','NOM_IRIS']].copy()
        diagnostic['NUMERATEUR'] = n.loc[mauvais]; diagnostic['DENOMINATEUR'] = d.loc[mauvais]
        exporter_csv(diagnostic, DOSSIER_INTERIM / ('anomalie_'+normaliser_nom(nom)+'.csv'))
        raise ValueError(f'Effectifs incohérents : {nom}. Examiner le diagnostic.')

def partition(colonnes, total, nom):
    somme = profil[colonnes].sum(axis=1,min_count=len(colonnes))
    verifier_effectif(somme,profil[total],nom,egalite=True)

def ajouter_ratio(colonne, numerateur, denominateur, libelle, theme, facteur=100):
    if facteur==100:
        verifier_effectif(profil[numerateur],profil[denominateur],colonne)
    RATIOS[colonne] = (numerateur,denominateur,facteur)
    profil[colonne] = profil[numerateur]/profil[denominateur].where(profil[denominateur]>0)*facteur
    enregistrer_indicateur({'COLONNE':colonne,'LIBELLE':libelle,'THEME':theme,
        'TYPE':'RATIO','NB_IRIS_CALCULABLES':int(profil[colonne].notna().sum()),
        'UNITE':'%' if facteur==100 else 'personnes par ménage',
        'NUMERATEUR':numerateur,'DENOMINATEUR':denominateur,'FACTEUR':facteur,
        'AGREGATION':'RECALCUL_SUR_SOMMES_COMPLETES','ANNEE':ANNEE_RP,'GEOGRAPHIE_SOURCE':GEOGRAPHIE_RP})

profil['J13B_STATUT_PCS_SOURCE'] = STATUT_PCS_SOURCE
profil['J13B_PCS_SERIE_COMPLETE'] = STATUT_PCS_SOURCE=='COMPLETE'


In [13]:
#Activité et catégorie socio-professionnelle des résidents

population_totale = nombre(profil['J13A_POPULATION'])
for col in ['J13B_POP_15_64','J13B_POP_15P_STATUT_CONJUGAL']+['J13B_POP_'+a for a in AGES_SCOL]:
    verifier_effectif(profil[col],population_totale,col+' / population J13a')
partition(['J13B_NB_ACTIFS_OCCUPES_15_64_P','J13B_NB_CHOMEURS_15_64'],
          'J13B_NB_ACTIFS_15_64','Actifs occupés et chômeurs')
partition(['J13B_NB_ACTIFS_15_64','J13B_NB_INACTIFS_15_64'],
          'J13B_POP_15_64','Actifs et inactifs')
ajouter_ratio('J13B_TAUX_ACTIVITE_15_64_PCT','J13B_NB_ACTIFS_15_64','J13B_POP_15_64',
              'Taux d’activité des 15–64 ans','ACTIVITE')
ajouter_ratio('J13B_TAUX_EMPLOI_RESIDENTS_15_64_PCT','J13B_NB_ACTIFS_OCCUPES_15_64_P','J13B_POP_15_64',
              'Taux d’emploi des résidents de 15–64 ans','ACTIVITE')
ajouter_ratio('J13B_PART_CHOMEURS_PARMI_ACTIFS_15_64_PCT','J13B_NB_CHOMEURS_15_64','J13B_NB_ACTIFS_15_64',
              'Part de chômeurs parmi les actifs de 15–64 ans — recensement','ACTIVITE')
ajouter_ratio('J13B_PART_ELEVES_ETUDIANTS_STAGIAIRES_15_64_PCT','J13B_NB_ELEVES_ETUDIANTS_STAGIAIRES_15_64','J13B_POP_15_64',
              'Élèves, étudiants et stagiaires non rémunérés parmi les 15–64 ans','ACTIVITE')
for suffixe, libelle in PCS.values():
    ajouter_ratio('J13B_PART_'+suffixe+'_PARMI_ACTIFS_OCCUPES_15_64_PCT',
        'J13B_NB_ACTIFS_OCCUPES_'+suffixe+'_15_64','J13B_NB_ACTIFS_OCCUPES_15_64_C',
        libelle+' parmi les actifs occupés résidents de 15–64 ans','ACTIVITE')



In [14]:
#Lieu de travail  et mode de déplacement des résidents

partition(['J13B_NB_ACTIFS_TRAVAIL_COMMUNE_RESIDENCE','J13B_NB_ACTIFS_TRAVAIL_AUTRE_COMMUNE'],
          'J13B_NB_ACTIFS_OCCUPES_15P_P','Lieu de travail des résidents')
for suffixe, source, libelle in [
 ('TRAVAIL_COMMUNE_RESIDENCE','J13B_NB_ACTIFS_TRAVAIL_COMMUNE_RESIDENCE','Travaillant dans leur commune'),
 ('TRAVAIL_AUTRE_COMMUNE','J13B_NB_ACTIFS_TRAVAIL_AUTRE_COMMUNE','Travaillant dans une autre commune')]:
    ajouter_ratio('J13B_PART_ACTIFS_'+suffixe+'_PCT',source,'J13B_NB_ACTIFS_OCCUPES_15P_P',
                  libelle+' parmi les actifs occupés résidents de 15 ans ou plus','MOBILITE_RESIDENTS')
MODES = {'SANS_TRANSPORT':'Sans transport','MARCHE':'Marche','VELO':'Vélo',
         'DEUX_ROUES_MOTORISES':'Deux-roues motorisé','VOITURE':'Voiture ou camion',
         'TRANSPORTS_COMMUN':'Transports en commun'}
partition(['J13B_NB_ACTIFS_'+s for s in MODES], 'J13B_NB_ACTIFS_OCCUPES_15P_C','Modes de transport des résidents')
for suffixe, libelle in MODES.items():
    ajouter_ratio('J13B_PART_ACTIFS_'+suffixe+'_PCT','J13B_NB_ACTIFS_'+suffixe,'J13B_NB_ACTIFS_OCCUPES_15P_C',
                  libelle+' pour aller travailler — actifs occupés résidents de 15 ans ou plus','MOBILITE_RESIDENTS')




In [15]:
#Scolarisation et diplomes

for age, libelle in AGES_SCOL.items():
    ajouter_ratio('J13B_PART_SCOLARISES_'+age+'_PCT','J13B_NB_SCOLARISES_'+age,'J13B_POP_'+age,
                  'Personnes scolarisées parmi les '+libelle,'FORMATION')
cols_diplomes = ['J13B_NB_NON_SCOLARISES_15P_'+s for s,_ in DIPLOMES.values()]
partition(cols_diplomes,'J13B_NB_NON_SCOLARISES_15P','Diplômes des non-scolarisés de 15 ans ou plus')
for suffixe, libelle in DIPLOMES.values():
    ajouter_ratio('J13B_PART_'+suffixe+'_PARMI_NON_SCOLARISES_15P_PCT',
                  'J13B_NB_NON_SCOLARISES_15P_'+suffixe,'J13B_NB_NON_SCOLARISES_15P',
                  libelle+' parmi les non-scolarisés de 15 ans ou plus','FORMATION')
profil['J13B_NB_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P'] = profil[
    ['J13B_NB_NON_SCOLARISES_15P_BAC_PLUS_2','J13B_NB_NON_SCOLARISES_15P_BAC_PLUS_3_4',
     'J13B_NB_NON_SCOLARISES_15P_BAC_PLUS_5_OU_PLUS']].sum(axis=1,min_count=3)
if 'J13B_NB_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P' not in COLONNES_EFFECTIFS:
    COLONNES_EFFECTIFS.append('J13B_NB_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P')
enregistrer_indicateur({'COLONNE':'J13B_NB_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P',
 'LIBELLE':'Diplômés du supérieur parmi les non-scolarisés de 15 ans ou plus','THEME':'FORMATION',
 'TYPE':'EFFECTIF_DERIVE','UNITE':'personnes','NUMERATEUR':'SUP2 + SUP34 + SUP5','DENOMINATEUR':'',
 'FACTEUR':1,'AGREGATION':'SOMME_SI_COMPLETE','ANNEE':ANNEE_RP,'GEOGRAPHIE_SOURCE':GEOGRAPHIE_RP})
ajouter_ratio('J13B_PART_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P_PCT',
 'J13B_NB_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P','J13B_NB_NON_SCOLARISES_15P',
 'Diplômés du supérieur parmi les non-scolarisés de 15 ans ou plus','FORMATION')


In [16]:
#Ménages, couples, familles et statut conjugal

partition(['J13B_NB_MENAGES_UNE_PERSONNE','J13B_NB_MENAGES_AUTRES_SANS_FAMILLE','J13B_NB_MENAGES_AVEC_FAMILLE'],
          'J13B_NB_MENAGES_C','Types de ménages')
partition(['J13B_NB_MENAGES_HOMME_SEUL','J13B_NB_MENAGES_FEMME_SEULE'],
          'J13B_NB_MENAGES_UNE_PERSONNE','Ménages d’une personne par sexe')
types_menages = {'UNE_PERSONNE':'Ménages d’une personne','AUTRES_SANS_FAMILLE':'Autres ménages sans famille',
    'AVEC_FAMILLE':'Ménages avec famille(s)','COUPLE_SANS_ENFANT':'Ménages avec couple sans enfant',
    'COUPLE_AVEC_ENFANTS':'Ménages avec couple avec enfant(s)',
    'FAMILLE_MONOPARENTALE':'Ménages avec famille monoparentale'}
for suffixe, libelle in types_menages.items():
    ajouter_ratio('J13B_PART_MENAGES_'+suffixe+'_PCT','J13B_NB_MENAGES_'+suffixe,'J13B_NB_MENAGES_C',
                  libelle+' parmi tous les ménages','MENAGES')
familles = ['COUPLE_SANS_ENFANT','COUPLE_AVEC_ENFANTS','FAMILLE_MONOPARENTALE']
partition(['J13B_NB_MENAGES_'+s for s in familles],'J13B_NB_MENAGES_AVEC_FAMILLE','Famille principale des ménages')
for suffixe in familles:
    ajouter_ratio('J13B_PART_'+suffixe+'_PARMI_MENAGES_AVEC_FAMILLE_PCT','J13B_NB_MENAGES_'+suffixe,
                  'J13B_NB_MENAGES_AVEC_FAMILLE',types_menages[suffixe]+' parmi les ménages avec famille','MENAGES')
ajouter_ratio('J13B_TAILLE_MOYENNE_MENAGES','J13B_POP_MENAGES_C','J13B_NB_MENAGES_C',
              'Nombre moyen de personnes par ménage','MENAGES',facteur=1)
partition(['J13B_NB_PERSONNES_15P_'+s for s,_ in CONJUGAL.values()],
          'J13B_POP_15P_STATUT_CONJUGAL','Statut conjugal des personnes de 15 ans ou plus')
for suffixe, libelle in CONJUGAL.values():
    ajouter_ratio('J13B_PART_PERSONNES_15P_'+suffixe+'_PCT','J13B_NB_PERSONNES_15P_'+suffixe,
                  'J13B_POP_15P_STATUT_CONJUGAL',libelle+' parmi les personnes de 15 ans ou plus','MENAGES')



In [17]:
#Logement, occupation et voiture

partition(['J13B_NB_RESIDENCES_PRINCIPALES','J13B_NB_RESIDENCES_SECONDAIRES_OCCASIONNELLES','J13B_NB_LOGEMENTS_VACANTS'],
          'J13B_NB_LOGEMENTS','Catégories de logements')
partition(['J13B_NB_RP_PROPRIETAIRES','J13B_NB_RP_LOCATAIRES','J13B_NB_RP_OCCUPATION_GRATUITE'],
          'J13B_NB_RESIDENCES_PRINCIPALES','Statut d’occupation des résidences principales')
verifier_effectif(profil['J13B_NB_RP_HLM_LOUE_VIDE'],profil['J13B_NB_RP_LOCATAIRES'],'HLM inclus dans les locataires')
for suffixe, libelle in {'RESIDENCES_PRINCIPALES':'Résidences principales',
    'RESIDENCES_SECONDAIRES_OCCASIONNELLES':'Résidences secondaires et logements occasionnels',
    'LOGEMENTS_VACANTS':'Logements vacants','MAISONS':'Maisons','APPARTEMENTS':'Appartements'}.items():
    ajouter_ratio('J13B_PART_'+suffixe+'_PCT','J13B_NB_'+suffixe,'J13B_NB_LOGEMENTS',
                  libelle+' parmi les logements','LOGEMENT')
for suffixe, libelle in {'PROPRIETAIRES':'Propriétaires','LOCATAIRES':'Locataires',
    'HLM_LOUE_VIDE':'HLM loués vides','OCCUPATION_GRATUITE':'Occupation gratuite'}.items():
    ajouter_ratio('J13B_PART_RP_'+suffixe+'_PCT','J13B_NB_RP_'+suffixe,'J13B_NB_RESIDENCES_PRINCIPALES',
                  libelle+' parmi les résidences principales','LOGEMENT')
partition(['J13B_NB_MENAGES_UNE_VOITURE','J13B_NB_MENAGES_DEUX_VOITURES_OU_PLUS'],
          'J13B_NB_MENAGES_AU_MOINS_UNE_VOITURE','Ménages avec voiture')
ajouter_ratio('J13B_PART_MENAGES_AU_MOINS_UNE_VOITURE_PCT','J13B_NB_MENAGES_AU_MOINS_UNE_VOITURE','J13B_NB_MENAGES_P',
              'Ménages disposant d’au moins une voiture','LOGEMENT')
ajouter_ratio('J13B_PART_MENAGES_EMMENAGES_MOINS_2_ANS_PCT','J13B_NB_MENAGES_EMMENAGES_MOINS_2_ANS','J13B_NB_MENAGES_P',
              'Ménages ayant emménagé depuis moins de deux ans','LOGEMENT')



In [18]:
#Controler et documenter les indicateurs RP

exporter_csv(pd.DataFrame(CONTROLES_EFFECTIFS),DOSSIER_INTERIM/'controle_effectifs_j13b.csv')
exporter_csv(pd.DataFrame(QUALITE_VARIABLES),DOSSIER_INTERIM/'qualite_variables_rp_j13b.csv')
for col,(n,d,facteur) in RATIOS.items():
    vals = profil[col].dropna()
    if not np.isfinite(vals.astype(float)).all():
        raise ValueError(f'Ratio non fini : {col}')
    if facteur==100 and not vals.between(0,100*(1+TOLERANCE_RELATIVE)+1e-6).all():
        raise ValueError(f'Part hors limites : {col}')
profil['J13B_ANNEE_RP'] = ANNEE_RP
profil['J13B_GEOGRAPHIE_RP'] = GEOGRAPHIE_RP
profil['J13B_NB_VARIABLES_RP_MANQUANTES'] = profil[COLONNES_EFFECTIFS].isna().sum(axis=1)
display(profil[['IRIS','NOM_IRIS','NOM_COMMUNE_ETUDE','J13B_TAUX_EMPLOI_RESIDENTS_15_64_PCT',
    'J13B_PART_MENAGES_UNE_PERSONNE_PCT','J13B_PART_SCOLARISES_1824_PCT',
    'J13B_PART_ACTIFS_TRANSPORTS_COMMUN_PCT','J13B_NB_VARIABLES_RP_MANQUANTES']].head(15))

disponibilite = pd.DataFrame(DIAGNOSTIC_COLONNES)
absentes_globales = disponibilite.loc[~disponibilite['PRESENTE']].copy()
exporter_csv(absentes_globales,DOSSIER_INTERIM/'colonnes_absentes_j13b.csv')
print('Colonnes sources absentes :',len(absentes_globales))
if not absentes_globales.empty:
    display(absentes_globales[['THEME','VARIABLE_SOURCE','COLONNE_SORTIE']])
couverture_indicateurs = pd.DataFrame([
    {'COLONNE':col,'NB_IRIS_CALCULABLES':int(profil[col].notna().sum()),
     'NB_IRIS_NON_CALCULABLES':int(profil[col].isna().sum())} for col in RATIOS])
exporter_csv(couverture_indicateurs,DOSSIER_INTERIM/'couverture_indicateurs_j13b.csv')
display(couverture_indicateurs)



,IRIS,NOM_IRIS,NOM_COMMUNE_ETUDE,J13B_TAUX_EMPLOI_RESIDENTS_15_64_PCT,J13B_PART_MENAGES_UNE_PERSONNE_PCT,J13B_PART_SCOLARISES_1824_PCT,J13B_PART_ACTIFS_TRANSPORTS_COMMUN_PCT,J13B_NB_VARIABLES_RP_MANQUANTES
0,751031101,Les Archives 1,Paris,75.250016,49.877062,71.467081,59.248457,0
1,751051702,Saint-Victor 2,Paris,70.409417,58.765669,87.874323,55.190338,0
2,751062309,Notre-Dame des Champs 9,Paris,57.712323,65.419456,92.880299,62.397916,0
3,751072503,Saint-Thomas d'Aquin 3,Paris,65.839923,52.760854,86.172231,32.615479,0
4,751135029,Gare 29,Paris,55.319483,37.880509,63.417613,63.350575,0
5,751135104,Maison Blanche 4,Paris,66.592353,66.554821,82.089081,73.762123,0
6,751135206,Croulebarbe 6,Paris,70.375595,51.4168,70.12981,59.191867,0
7,751145512,Petit Montrouge 12,Paris,66.444285,58.985524,87.523896,64.536034,0
8,751187008,Clignancourt 8,Paris,79.139253,54.933423,75.876799,50.017278,0
9,751187017,Clignancourt 17,Paris,71.677572,61.401942,54.401622,68.020161,0


Colonnes sources absentes : 0


,COLONNE,NB_IRIS_CALCULABLES,NB_IRIS_NON_CALCULABLES
0,J13B_TAUX_ACTIVITE_15_64_PCT,5208,56
1,J13B_TAUX_EMPLOI_RESIDENTS_15_64_PCT,5208,56
2,J13B_PART_CHOMEURS_PARMI_ACTIFS_15_64_PCT,5207,57
3,J13B_PART_ELEVES_ETUDIANTS_STAGIAIRES_15_64_PCT,5208,56
4,J13B_PART_AGRICULTEURS_PARMI_ACTIFS_OCCUPES_15...,5198,66
...,...,...,...
55,J13B_PART_RP_LOCATAIRES_PCT,5198,66
56,J13B_PART_RP_HLM_LOUE_VIDE_PCT,5198,66
57,J13B_PART_RP_OCCUPATION_GRATUITE_PCT,5198,66
58,J13B_PART_MENAGES_AU_MOINS_UNE_VOITURE_PCT,5198,66


In [19]:
#Préparer les revenus disponibles

COL_MED = 'J13B_NIVEAU_VIE_MEDIAN_ANNUEL_UC_EUR'
COL_PAUV = 'J13B_TAUX_PAUVRETE_60_PCT'
revenus = pd.DataFrame(columns=['IRIS',COL_MED,COL_PAUV])
ALIAS_MED = {'MED'+YY_FILO,'DISP_MED'+YY_FILO}
ALIAS_PAUV = {'TP60'+YY_FILO,'DISP_TP60'+YY_FILO}
ALIAS_NOTE = {'NOTE_DISP','NOTE_DISP'+YY_FILO,'DISP_NOTE'+YY_FILO,'NOTE'}
if CHARGER_REVENUS:
    revenus = lire_table(FICHIER_REVENUS,[ALIAS_IRIS,ALIAS_MED,ALIAS_PAUV],
        ALIAS_IRIS | ALIAS_COM | ALIAS_MED | ALIAS_PAUV | ALIAS_NOTE | {'NOTE_DECL'},
        MEMBRES['REVENUS'],FEUILLES['REVENUS'])
    if 'DECL' in FICHIER_REVENUS.name.upper() or any(c.startswith('DECL_') or c=='NOTE_DECL' for c in revenus.columns):
        raise ValueError('Utiliser les revenus DISPONIBLES, pas les revenus déclarés.')
    ci = choisir_colonne(revenus,ALIAS_IRIS,'Revenus — IRIS')
    cm = choisir_colonne(revenus,ALIAS_MED,'Revenus — médiane')
    cp = choisir_colonne(revenus,ALIAS_PAUV,'Revenus — pauvreté')
    revenus = verifier_cle(revenus.rename(columns={ci:'IRIS'}),'IRIS',9,'Filosofi IRIS')
    revenus = revenus[revenus['IRIS'].str[:2].isin(DEPARTEMENTS_IDF)].copy()
    erreurs_revenus = []
    for source,destination in [(cm,COL_MED),(cp,COL_PAUV)]:
        revenus[destination] = nombre(revenus[source]).where(renseigne(revenus[source]))
        mauvais = renseigne(revenus[source]) & revenus[destination].isna()
        if destination==COL_PAUV:
            mauvais |= (revenus[destination].notna() & ~revenus[destination].between(0,100))
        for index in revenus.index[mauvais]:
            erreurs_revenus.append({'IRIS':revenus.at[index,'IRIS'],'VARIABLE':source,'VALEUR':str(revenus.at[index,source])})
        revenus[destination+'_DISPONIBLE'] = revenus[destination].notna()
    exporter_csv(pd.DataFrame(erreurs_revenus,columns=['IRIS','VARIABLE','VALEUR']),
                  DOSSIER_INTERIM/'anomalies_revenus_j13b.csv')
    if erreurs_revenus:
        raise ValueError('Valeurs de revenus invalides : consulter anomalies_revenus_j13b.csv.')
    revenus['J13B_ANNEE_FILOSOFI_SOURCE'] = ANNEE_FILOSOFI
    revenus['J13B_GEOGRAPHIE_FILOSOFI_SOURCE'] = GEOGRAPHIE_FILOSOFI
    revenus['J13B_IRIS_REVENUS_SOURCE'] = revenus['IRIS']
    exporter_csv(revenus,DOSSIER_PROCESSED/f'revenus_iris_idf_filosofi{ANNEE_FILOSOFI}_geo{GEOGRAPHIE_FILOSOFI}_j13b.csv')
    print('IRIS présents dans la source fiscale :',len(revenus))
    display(revenus[['IRIS',COL_MED,COL_PAUV]].head(10))
# Un niveau de vie médian peut être nul ou négatif ; ce n’est pas un effectif.

if CHARGER_REVENUS and revenus.empty:
    raise ValueError('Aucune ligne fiscale IDF lue : vérifier le fichier de revenus et ses codes IRIS.')
if not CHARGER_REVENUS:
    print('Revenus désactivés : CHARGER_REVENUS=False. Passer à True au bloc 4 puis relancer depuis le début.')




Lecture : BASE_TD_FILO_IRIS_2021_DISP_CSV.zip {'membre': 'BASE_TD_FILO_IRIS_2021_DISP.csv', 'feuille': None, 'header': 0, 'sep': ';', 'encoding': 'utf-8-sig', 'format': 'csv'}
IRIS présents dans la source fiscale : 4329


,IRIS,J13B_NIVEAU_VIE_MEDIAN_ANNUEL_UC_EUR,J13B_TAUX_PAUVRETE_60_PCT
9762,751010101,<NA>,<NA>
9763,751010102,<NA>,<NA>
9764,751010103,<NA>,<NA>
9765,751010104,<NA>,<NA>
9766,751010105,<NA>,<NA>
9767,751010199,<NA>,<NA>
9768,751010201,31290.0,15.0
9769,751010202,38680.0,13.0
9770,751010203,32760.0,11.0
9771,751010204,31480.0,12.0


In [20]:
#Définir les correspondances fiscales

colonnes_passage = ['IRIS_SOURCE','IRIS_CIBLE','GEOGRAPHIE_SOURCE','GEOGRAPHIE_CIBLE',
                    'TYPE_CORRESPONDANCE','SOURCE_VALIDATION','VALIDE']
passage = pd.DataFrame(columns=colonnes_passage+['METHODE_JOINTURE','CONTOUR_VERIFIE'])
SOURCES_REFERENCE_REVENUS = []
diagnostic_reference_revenus = pd.DataFrame(columns=['IRIS','ETAT'])

def obtenir_reference_iris(fichier, annee):
    fichier = Path(fichier)
    urls = {
        2022:'https://www.insee.fr/fr/statistiques/fichier/7708995/reference_IRIS_geo2022.zip',
        2024:'https://www.insee.fr/fr/statistiques/fichier/7708995/reference_IRIS_geo2024.zip',
    }
    if fichier.is_file():
        return fichier
    if not TELECHARGER_REFERENCES_ABSENTES or annee not in urls:
        raise FileNotFoundError(f'Référence IRIS {annee} absente : {fichier}. '
            'Télécharger la table de référence sur https://www.insee.fr/fr/information/7708995 '
            'ou renseigner FICHIER_CORRESPONDANCE_REVENUS.')
    from urllib.request import urlopen
    fichier.parent.mkdir(parents=True,exist_ok=True)
    temporaire = fichier.with_suffix(fichier.suffix+'.part')
    print(f'Téléchargement de la référence Insee IRIS {annee}…')
    try:
        with urlopen(urls[annee],timeout=30) as entree, temporaire.open('wb') as sortie:
            while True:
                bloc = entree.read(1024*1024)
                if not bloc:
                    break
                sortie.write(bloc)
        with ZipFile(temporaire) as archive:
            if archive.testzip() is not None or not any(n.lower().endswith('.xlsx') for n in archive.namelist()):
                raise ValueError('Archive de référence IRIS invalide ou incomplète.')
        temporaire.replace(fichier)
    except Exception as erreur:
        temporaire.unlink(missing_ok=True)
        raise RuntimeError(f'Référence IRIS {annee} non téléchargée. Placer le ZIP officiel '
            f'dans {fichier} puis relancer le bloc 17. Détail : {erreur}') from erreur
    return fichier

def verifier_millesime_reference(fichier, info, annee, feuille=None):
    with ouvrir_binaire(fichier,info['membre']) as flux:
        apercu = pd.read_excel(io.BytesIO(flux.read()),sheet_name=feuille or info['feuille'],
                              header=None,nrows=5,dtype='string',engine='openpyxl')
    textes = ' '.join(apercu.stack().dropna().astype(str))
    annees = {int(a) for a in re.findall(r'(?:01/01/|1er janvier\s+)(\d{4})',textes)}
    if annees != {annee}:
        raise ValueError(f'Référence IRIS : géographie {annee} attendue, dates lues : {sorted(annees)}. '
            'Vérifier le contenu du fichier, pas seulement son nom.')

def lire_reference_iris(fichier, annee):
    groupes = [{'CODE_IRIS'},{'DEPCOM'},{'TYP_IRIS'}]
    info = detecter_table(fichier,groupes,feuille='Emboitements_IRIS')
    verifier_millesime_reference(fichier,info,annee)
    table = lire_table(fichier,groupes,{'CODE_IRIS','DEPCOM','TYP_IRIS'},info=info)
    table = verifier_cle(table.rename(columns={'CODE_IRIS':'IRIS'}),'IRIS',9,'Référence IRIS '+str(annee))
    table['DEPCOM'] = normaliser_code(table['DEPCOM'],5)
    table['TYP_IRIS'] = table['TYP_IRIS'].astype('string').str.strip().str.upper()
    if not table['TYP_IRIS'].isin(['H','A','D','Z']).all():
        raise ValueError('Référence IRIS : type absent ou non reconnu.')
    return table[table['IRIS'].str[:2].isin(DEPARTEMENTS_IDF)].copy(), info

def correspondances_codes_stables(fichier_source, fichier_cible):
    # La compatibilité porte sur les codes et les changements répertoriés.
    # La documentation Insee précise que les petits échanges de parcelles ne sont pas exhaustifs.
    # Ce traitement ne certifie donc PAS une égalité géométrique des contours.
    if not 0 < GEOGRAPHIE_RP-GEOGRAPHIE_FILOSOFI <= 4:
        raise ValueError('Le contrôle automatique exige une géographie cible postérieure '
            'et un écart de quatre ans au plus, couvert par l’historique sur cinq ans.')
    source, info_source = lire_reference_iris(fichier_source,GEOGRAPHIE_FILOSOFI)
    cible, info_cible = lire_reference_iris(fichier_cible,GEOGRAPHIE_RP)
    groupes_modifications = [{'ANNEE_MODIF'},{'IRIS_INI'},{'IRIS_FIN'},{'MODIF_IRIS'}]
    info_mod = detecter_table(fichier_cible,groupes_modifications,feuille='Modifications_IRIS')
    verifier_millesime_reference(fichier_cible,info_mod,GEOGRAPHIE_RP)
    with ouvrir_binaire(fichier_cible,info_mod['membre']) as flux:
        contenu = io.BytesIO(flux.read())
    documentation = pd.read_excel(contenu,sheet_name='Documentation',header=None,dtype='string',engine='openpyxl')
    texte_documentation = ' '.join(documentation.stack().dropna().astype(str))
    if not re.search(r'(?:cinq|5)\s+ans',texte_documentation,flags=re.IGNORECASE):
        raise ValueError('Historique IRIS : couverture de cinq ans non documentée.')
    modifications = lire_table(fichier_cible,groupes_modifications,
        {'ANNEE_MODIF','IRIS_INI','IRIS_FIN','MODIF_IRIS','NATURE_MODIF'},info=info_mod)
    annees = nombre(modifications['ANNEE_MODIF'])
    if not annees.notna().all() or not annees.eq(annees.round()).all() or annees.gt(GEOGRAPHIE_RP).any():
        raise ValueError('Historique IRIS : années de modification invalides.')
    intervalle = annees.gt(GEOGRAPHIE_FILOSOFI) & annees.le(GEOGRAPHIE_RP)
    modif = modifications[intervalle & ~modifications['MODIF_IRIS'].astype('string').str.strip().eq('0')].copy()
    touches = set()
    for col in ['IRIS_INI','IRIS_FIN']:
        codes = normaliser_code(modif[col],9).dropna()
        touches.update(codes[codes.str[:2].isin(DEPARTEMENTS_IDF)])
    exporter_csv(modif,DOSSIER_INTERIM/'modifications_iris_intervalle_revenus_j13b.csv')
    exporter_csv(pd.DataFrame({'DOCUMENTATION_INSEE':[texte_documentation]}),
                 DOSSIER_INTERIM/'documentation_reference_revenus_j13b.csv')

    communs = source.merge(cible,on='IRIS',how='inner',validate='one_to_one',suffixes=('_SOURCE','_CIBLE'))
    communs = communs.set_index('IRIS')
    codes_source = set(source['IRIS'])
    codes_cible = set(cible['IRIS'])
    etats = []
    for iris in profil['IRIS']:
        if iris not in codes_cible:
            etat = 'CODE_ABSENT_REFERENCE_CIBLE'
        elif iris not in codes_source:
            etat = 'CODE_ABSENT_REFERENCE_SOURCE'
        elif iris in touches:
            etat = 'MODIFICATION_IRIS_REPERTORIEE'
        elif communs.at[iris,'TYP_IRIS_SOURCE']=='Z' or communs.at[iris,'TYP_IRIS_CIBLE']=='Z':
            etat = 'COMMUNE_NON_IRISEE_A_VALIDER'
        elif communs.at[iris,'DEPCOM_SOURCE'] != communs.at[iris,'DEPCOM_CIBLE']:
            etat = 'COMMUNE_SOURCE_CIBLE_DIFFERENTE'
        elif communs.at[iris,'TYP_IRIS_SOURCE'] != communs.at[iris,'TYP_IRIS_CIBLE']:
            etat = 'TYPE_IRIS_SOURCE_CIBLE_DIFFERENT'
        else:
            etat = 'CODE_STABLE_HORS_MODIFICATIONS_REPERTORIEES'
        etats.append({'IRIS':iris,'ETAT':etat})
    diagnostic = pd.DataFrame(etats)
    stables = diagnostic.loc[diagnostic['ETAT'].eq('CODE_STABLE_HORS_MODIFICATIONS_REPERTORIEES'),'IRIS']
    preuve = (f'Insee : références IRIS {GEOGRAPHIE_FILOSOFI} et {GEOGRAPHIE_RP}, '
        'feuilles Emboitements_IRIS et Modifications_IRIS ; '
        'https://www.insee.fr/fr/information/7708995 ; '
        'contrôle de nomenclature, petits échanges de parcelles non exhaustifs')
    correspondances = pd.DataFrame({'IRIS_SOURCE':stables,'IRIS_CIBLE':stables,
        'GEOGRAPHIE_SOURCE':GEOGRAPHIE_FILOSOFI,'GEOGRAPHIE_CIBLE':GEOGRAPHIE_RP,
        'TYPE_CORRESPONDANCE':'CODE_STABLE_HORS_MODIFICATIONS_REPERTORIEES',
        'SOURCE_VALIDATION':preuve,'VALIDE':True,
        'METHODE_JOINTURE':'REFERENCES_INSEE_CODE_STABLE','CONTOUR_VERIFIE':False})
    return correspondances, diagnostic

if CHARGER_REVENUS and GEOGRAPHIE_FILOSOFI==GEOGRAPHIE_RP:
    passage = pd.DataFrame({'IRIS_SOURCE':profil['IRIS'],'IRIS_CIBLE':profil['IRIS'],
        'GEOGRAPHIE_SOURCE':GEOGRAPHIE_FILOSOFI,'GEOGRAPHIE_CIBLE':GEOGRAPHIE_RP,
        'TYPE_CORRESPONDANCE':'MEME_GEOGRAPHIE','SOURCE_VALIDATION':'Géographies déclarées identiques','VALIDE':True,
        'METHODE_JOINTURE':'MEME_GEOGRAPHIE_SOURCE','CONTOUR_VERIFIE':True})
elif CHARGER_REVENUS and FICHIER_CORRESPONDANCE_REVENUS is not None:
    fichier_passage = Path(FICHIER_CORRESPONDANCE_REVENUS)
    if not fichier_passage.is_file():
        raise FileNotFoundError(f'Correspondance fiscale absente : {fichier_passage}')
    passage = entetes(pd.read_csv(fichier_passage,sep=None,engine='python',dtype='string',encoding='utf-8-sig'))
    if set(colonnes_passage)-set(passage.columns):
        raise ValueError('Correspondance fiscale : colonnes obligatoires absentes.')
    passage['VALIDE'] = booleen(passage['VALIDE'],'Correspondance — VALIDE')
    geo_source = nombre(passage['GEOGRAPHIE_SOURCE']); geo_cible = nombre(passage['GEOGRAPHIE_CIBLE'])
    if (not geo_source.notna().all() or not geo_cible.notna().all()
            or not geo_source.eq(GEOGRAPHIE_FILOSOFI).all() or not geo_cible.eq(GEOGRAPHIE_RP).all()):
        raise ValueError('Géographies du fichier de correspondance incompatibles.')
    passage = passage[passage['VALIDE']].copy()
    passage = verifier_cle(passage,'IRIS_SOURCE',9,'Passage — source')
    passage = verifier_cle(passage,'IRIS_CIBLE',9,'Passage — cible')
    types = passage['TYPE_CORRESPONDANCE'].astype('string').str.strip().str.upper()
    preuves = passage['SOURCE_VALIDATION'].astype('string').str.strip().fillna('')
    if not types.isin(['INCHANGE','RENOMMAGE_SANS_CHANGEMENT_DE_CONTOUR']).all() or preuves.eq('').any():
        raise ValueError('Correspondance fiscale : contour modifié ou validation non documentée.')
    if set(passage['IRIS_CIBLE'])-ATTENDUS or not passage['IRIS_SOURCE'].str[:2].isin(DEPARTEMENTS_IDF).all():
        raise ValueError('Correspondance fiscale hors périmètre IDF/J13a.')
    passage['METHODE_JOINTURE'] = 'CORRESPONDANCE_CONTOURS_DOCUMENTEE'
    passage['CONTOUR_VERIFIE'] = True
elif CHARGER_REVENUS and MODE_JOINTURE_REVENUS=='REFERENCES_INSEE':
    chemin_source = Path(FICHIER_REFERENCE_IRIS_REVENUS) if FICHIER_REFERENCE_IRIS_REVENUS is not None else (
        DOSSIER_REFERENCES/f'reference_IRIS_geo{GEOGRAPHIE_FILOSOFI}.zip')
    chemin_cible = Path(FICHIER_REFERENCE_IRIS_PROFIL) if FICHIER_REFERENCE_IRIS_PROFIL is not None else (
        DOSSIER_REFERENCES/f'reference_IRIS_geo{GEOGRAPHIE_RP}.zip')
    obtenir_reference_iris(chemin_source,GEOGRAPHIE_FILOSOFI)
    obtenir_reference_iris(chemin_cible,GEOGRAPHIE_RP)
    passage, diagnostic_reference_revenus = correspondances_codes_stables(chemin_source,chemin_cible)
    SOURCES_REFERENCE_REVENUS = [chemin_source,chemin_cible]
    print('Jointure sur codes stables contrôlés dans les références Insee. '
          'Cette méthode ne certifie pas une égalité géométrique des contours.')
else:
    fichier_modele = DOSSIER_INTERIM/'modele_correspondance_revenus_iris.csv'
    if CHARGER_REVENUS and not fichier_modele.exists():
        modele = pd.DataFrame({'IRIS_SOURCE':'','IRIS_CIBLE':profil['IRIS'],
            'GEOGRAPHIE_SOURCE':GEOGRAPHIE_FILOSOFI,'GEOGRAPHIE_CIBLE':GEOGRAPHIE_RP,
            'TYPE_CORRESPONDANCE':'','SOURCE_VALIDATION':'','VALIDE':False})
        exporter_csv(modele,fichier_modele)
        print('Mode VALIDATION_CONTOURS : renseigner FICHIER_CORRESPONDANCE_REVENUS. '
              'Modèle à vérifier :',fichier_modele)
exporter_csv(diagnostic_reference_revenus,DOSSIER_INTERIM/'diagnostic_reference_revenus_j13b.csv')
exporter_csv(passage,DOSSIER_INTERIM/'correspondances_revenus_utilisees_j13b.csv')
print('Correspondances utilisées :',len(passage))


Lecture : reference_IRIS_geo2022.zip {'membre': 'reference_IRIS_geo2022.xlsx', 'feuille': 'Emboitements_IRIS', 'header': 5, 'sep': None, 'encoding': None, 'format': 'xlsx'}
Lecture : reference_IRIS_geo2024.zip {'membre': 'reference_IRIS_geo2024.xlsx', 'feuille': 'Emboitements_IRIS', 'header': 5, 'sep': None, 'encoding': None, 'format': 'xlsx'}
Lecture : reference_IRIS_geo2024.zip {'membre': 'reference_IRIS_geo2024.xlsx', 'feuille': 'Modifications_IRIS', 'header': 5, 'sep': None, 'encoding': None, 'format': 'xlsx'}
Jointure sur codes stables contrôlés dans les références Insee. Cette méthode ne certifie pas une égalité géométrique des contours.
Correspondances utilisées : 4329


In [21]:
#Joindre les revenus compatibles et conserver leur statut

colonnes_fiscales = ['J13B_IRIS_REVENUS_SOURCE',COL_MED,COL_PAUV,
                    'J13B_ANNEE_FILOSOFI_SOURCE','J13B_GEOGRAPHIE_FILOSOFI_SOURCE']
# Retirer uniquement les anciennes sorties fiscales avant de refaire la jointure.
profil = profil.drop(columns=colonnes_fiscales+['J13B_STATUT_REVENUS',
    'J13B_MEDIANE_REVENUS_DISPONIBLE','J13B_TAUX_PAUVRETE_DISPONIBLE'],errors='ignore')
if not passage.empty:
    fiscal = revenus.rename(columns={'IRIS':'IRIS_SOURCE'})
    if 'J13B_IRIS_REVENUS_SOURCE' not in fiscal.columns:
        raise ValueError('Source fiscale non préparée.')
    joint = passage[['IRIS_SOURCE','IRIS_CIBLE']].merge(
        fiscal[['IRIS_SOURCE']+colonnes_fiscales],on='IRIS_SOURCE',how='left',validate='one_to_one',indicator='_FISCAL')
    joint['J13B_STATUT_REVENUS'] = np.where(joint['_FISCAL'].eq('both'),'LIGNE_FISCALE_PRESENTE','IRIS_SOURCE_NON_DIFFUSE')
    joint = joint.rename(columns={'IRIS_CIBLE':'IRIS'}).drop(columns=['IRIS_SOURCE','_FISCAL'])
    profil = profil.merge(joint,on='IRIS',how='left',validate='one_to_one')
else:
    for col in colonnes_fiscales:
        profil[col] = pd.NA
    profil['J13B_STATUT_REVENUS'] = pd.NA
profil['J13B_STATUT_REVENUS'] = profil['J13B_STATUT_REVENUS'].fillna(
    'GEOGRAPHIE_NON_HARMONISEE' if CHARGER_REVENUS else 'REVENUS_NON_CHARGES')
for col in [COL_MED,COL_PAUV]:
    profil[col] = nombre(profil[col])
profil['J13B_MEDIANE_REVENUS_DISPONIBLE'] = profil[COL_MED].notna()
profil['J13B_TAUX_PAUVRETE_DISPONIBLE'] = profil[COL_PAUV].notna()
ligne = profil['J13B_STATUT_REVENUS'].eq('LIGNE_FISCALE_PRESENTE')
deux = profil[[COL_MED,COL_PAUV]].notna().all(axis=1)
une = profil[[COL_MED,COL_PAUV]].notna().any(axis=1)
profil.loc[ligne & deux,'J13B_STATUT_REVENUS'] = 'DISPONIBLE'
profil.loc[ligne & une & ~deux,'J13B_STATUT_REVENUS'] = 'PARTIEL'
profil.loc[ligne & ~une,'J13B_STATUT_REVENUS'] = 'VALEURS_NON_DIFFUSEES_OU_MANQUANTES'
for col,libelle,unite in [(COL_MED,'Niveau de vie médian annuel par unité de consommation','euros annuels / UC'),
                          (COL_PAUV,'Taux de pauvreté au seuil de 60 %','%')]:
    enregistrer_indicateur({'COLONNE':col,'LIBELLE':libelle,'THEME':'REVENUS_DISPONIBLES',
        'TYPE':'INDICATEUR_FISCAL','UNITE':unite,'NUMERATEUR':'indicateur publié','DENOMINATEUR':'',
        'FACTEUR':1,'AGREGATION':'NON_AGREGABLE_ICI','ANNEE':ANNEE_FILOSOFI,'GEOGRAPHIE_SOURCE':GEOGRAPHIE_FILOSOFI})
statuts_revenus = profil['J13B_STATUT_REVENUS'].value_counts().rename_axis('STATUT').reset_index(name='NB_IRIS')
exporter_csv(statuts_revenus,DOSSIER_INTERIM/'statuts_revenus_j13b.csv')
exporter_csv(profil[['IRIS','J13B_IRIS_REVENUS_SOURCE','J13B_STATUT_REVENUS',
    'J13B_MEDIANE_REVENUS_DISPONIBLE','J13B_TAUX_PAUVRETE_DISPONIBLE']],
    DOSSIER_INTERIM/'couverture_revenus_j13b.csv')
display(statuts_revenus)
assert len(profil)==NOMBRE_IRIS_INITIAL and set(profil['IRIS'])==ATTENDUS



,STATUT,NB_IRIS
0,DISPONIBLE,3950
1,GEOGRAPHIE_NON_HARMONISEE,935
2,VALEURS_NON_DIFFUSEES_OU_MANQUANTES,376
3,PARTIEL,3


In [22]:
profil.head(20)

,IRIS,CODGEO_REFERENCE,CODGEO_SOURCE,CODGEO_ETUDE,DEPARTEMENT,NOM_IRIS,NOM_COMMUNE_SOURCE,TYP_IRIS,NOM_COMMUNE_REFERENCE,ORIGINE_SELECTION_J12,NOM_COMMUNE_ETUDE,J13A_COMMUNE_PRESELECTIONNEE_J12,J13A_DANS_FOCUS,J13A_SURFACE_KM2,CODGEO_POPULATION_SOURCE,P22_POP,P22_POPH,P22_POPF,P22_POP0014,P22_POP1529,P22_POP3044,P22_POP4559,P22_POP6074,P22_POP75P,TYP_IRIS_RP,J13A_POPULATION,J13A_POPULATION_HOMMES,J13A_POPULATION_FEMMES,J13A_POP_0014,J13A_POP_1529,J13A_POP_3044,J13A_POP_4559,J13A_POP_6074,J13A_POP_75P,J13A_PART_HOMMES_PCT,J13A_PART_FEMMES_PCT,J13A_PART_0014_PCT,J13A_PART_1529_PCT,J13A_PART_3044_PCT,J13A_PART_4559_PCT,J13A_PART_6074_PCT,J13A_PART_75P_PCT,J13A_DENSITE_POP_HAB_KM2,J13A_ANNEE_RP,J13A_GEOGRAPHIE,J13A_LIBELLE_TYPE_IRIS,J13A_POPULATION_DISPONIBLE,J13B_DANS_FOCUS,J13B_POP_15_64,J13B_NB_ACTIFS_15_64,...,J13B_PART_SCOLARISES_1824_PCT,J13B_PART_SCOLARISES_2529_PCT,J13B_PART_SCOLARISES_30P_PCT,J13B_PART_SANS_DIPLOME_CEP_PARMI_NON_SCOLARISES_15P_PCT,J13B_PART_BEPC_BREVET_PARMI_NON_SCOLARISES_15P_PCT,J13B_PART_CAP_BEP_PARMI_NON_SCOLARISES_15P_PCT,J13B_PART_BAC_PARMI_NON_SCOLARISES_15P_PCT,J13B_PART_BAC_PLUS_2_PARMI_NON_SCOLARISES_15P_PCT,J13B_PART_BAC_PLUS_3_4_PARMI_NON_SCOLARISES_15P_PCT,J13B_PART_BAC_PLUS_5_OU_PLUS_PARMI_NON_SCOLARISES_15P_PCT,J13B_NB_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P,J13B_PART_DIPLOMES_SUPERIEUR_NON_SCOLARISES_15P_PCT,J13B_PART_MENAGES_UNE_PERSONNE_PCT,J13B_PART_MENAGES_AUTRES_SANS_FAMILLE_PCT,J13B_PART_MENAGES_AVEC_FAMILLE_PCT,J13B_PART_MENAGES_COUPLE_SANS_ENFANT_PCT,J13B_PART_MENAGES_COUPLE_AVEC_ENFANTS_PCT,J13B_PART_MENAGES_FAMILLE_MONOPARENTALE_PCT,J13B_PART_COUPLE_SANS_ENFANT_PARMI_MENAGES_AVEC_FAMILLE_PCT,J13B_PART_COUPLE_AVEC_ENFANTS_PARMI_MENAGES_AVEC_FAMILLE_PCT,J13B_PART_FAMILLE_MONOPARENTALE_PARMI_MENAGES_AVEC_FAMILLE_PCT,J13B_TAILLE_MOYENNE_MENAGES,J13B_PART_PERSONNES_15P_MARIEES_PCT,J13B_PART_PERSONNES_15P_PACSEES_PCT,J13B_PART_PERSONNES_15P_CONCUBINAGE_PCT,J13B_PART_PERSONNES_15P_VEUVES_PCT,J13B_PART_PERSONNES_15P_DIVORCEES_PCT,J13B_PART_PERSONNES_15P_CELIBATAIRES_PCT,J13B_PART_RESIDENCES_PRINCIPALES_PCT,J13B_PART_RESIDENCES_SECONDAIRES_OCCASIONNELLES_PCT,J13B_PART_LOGEMENTS_VACANTS_PCT,J13B_PART_MAISONS_PCT,J13B_PART_APPARTEMENTS_PCT,J13B_PART_RP_PROPRIETAIRES_PCT,J13B_PART_RP_LOCATAIRES_PCT,J13B_PART_RP_HLM_LOUE_VIDE_PCT,J13B_PART_RP_OCCUPATION_GRATUITE_PCT,J13B_PART_MENAGES_AU_MOINS_UNE_VOITURE_PCT,J13B_PART_MENAGES_EMMENAGES_MOINS_2_ANS_PCT,J13B_ANNEE_RP,J13B_GEOGRAPHIE_RP,J13B_NB_VARIABLES_RP_MANQUANTES,J13B_IRIS_REVENUS_SOURCE,J13B_NIVEAU_VIE_MEDIAN_ANNUEL_UC_EUR,J13B_TAUX_PAUVRETE_60_PCT,J13B_ANNEE_FILOSOFI_SOURCE,J13B_GEOGRAPHIE_FILOSOFI_SOURCE,J13B_STATUT_REVENUS,J13B_MEDIANE_REVENUS_DISPONIBLE,J13B_TAUX_PAUVRETE_DISPONIBLE
0,751031101,75103,75103,75056,75,Les Archives 1,Paris 3e Arrondissement,H,Paris,<NA>,Paris,False,False,0.07692047000028567,75103,2317.00112634046,1124.52462230211,1192.47650403835,290.084949836359,624.497460051535,516.04310796355,465.951279225699,268.757081470598,151.66724779272,H,2317.00112634046,1124.52462230211,1192.47650403835,290.084949836359,624.497460051535,516.04310796355,465.951279225699,268.757081470598,151.66724779272,48.53362432664059,51.46637567335941,12.519845007349122,26.95283368454312,22.27202663378087,20.110101541540303,11.599350488667257,6.5458426441193724,30122.03547809647,2022,2024,Habitat,True,False,1713.930041,1405.239542,...,71.467081,19.343631,2.352776,8.25023,1.912987,3.62426,11.272267,5.868306,18.717076,50.354875,1264.512455,74.940256,49.877062,3.11585,47.007088,22.022415,14.617486,10.367187,46.849137,31.096345,22.054518,1.851384,30.591532,5.293667,12.344187,2.719037,6.470031,42.581546,81.533253,11.530828,6.935918,0.680788,98.963068,35.453309,57.260374,11.206262,7.286318,24.79181,16.090138,2022,2024,0,751031101,37610.0,12.0,2021.0,2022.0,DISPONIBLE,True,True
1,751051702,75105,75105,75056,75,Saint-Victor 2,Paris 5e Arrondissement,H,Paris,<NA>,Paris,False,False,0.048481215000010146,75105,1665.56495706618,771.558324143039,894.0066329231

In [23]:
#Synthèse des indicateurs RP

def somme_complete(serie):
    return float(serie.sum()) if serie.notna().all() else pd.NA
population_base = nombre(profil['J13A_POPULATION'])
profil['J13A_POPULATION'] = population_base
lignes = []
for code_commune,groupe in profil.groupby('CODGEO_ETUDE',sort=True):
    ligne = {'CODGEO_ETUDE':code_commune,'NOM_COMMUNE_ETUDE':groupe['NOM_COMMUNE_ETUDE'].iloc[0],
        'NB_IRIS':len(groupe),'J13A_COMMUNE_PRESELECTIONNEE_J12':bool(groupe['J13A_COMMUNE_PRESELECTIONNEE_J12'].iloc[0]),
        'J13B_DANS_FOCUS':bool(groupe['J13B_DANS_FOCUS'].iloc[0]),
        'J13A_POPULATION':somme_complete(groupe['J13A_POPULATION']),
        'NB_IRIS_MEDIANE_REVENUS_DISPONIBLE':int(groupe['J13B_MEDIANE_REVENUS_DISPONIBLE'].sum())}
    for col in COLONNES_EFFECTIFS:
        ligne[col] = somme_complete(groupe[col])
    lignes.append(ligne)
synthese = pd.DataFrame(lignes)
for col in ['J13A_POPULATION']+COLONNES_EFFECTIFS:
    synthese[col] = pd.array(synthese[col],dtype='Float64')
for col,(n,d,facteur) in RATIOS.items():
    synthese[col] = synthese[n]/synthese[d].where(synthese[d]>0)*facteur
assert COL_MED not in synthese.columns and COL_PAUV not in synthese.columns
display(synthese.loc[synthese['J13B_DANS_FOCUS'],['CODGEO_ETUDE','NOM_COMMUNE_ETUDE','NB_IRIS',
    'J13B_NB_MENAGES_C','J13B_PART_MENAGES_UNE_PERSONNE_PCT','J13B_PART_ACTIFS_TRANSPORTS_COMMUN_PCT']])


,CODGEO_ETUDE,NOM_COMMUNE_ETUDE,NB_IRIS,J13B_NB_MENAGES_C,J13B_PART_MENAGES_UNE_PERSONNE_PCT,J13B_PART_ACTIFS_TRANSPORTS_COMMUN_PCT
560,78146,Chatou,13,13021.218542,35.575431,42.810428
961,92002,Antony,27,27961.786741,36.511205,44.279018
967,92019,Châtenay-Malabry,14,14950.967415,33.948308,36.346969
982,92048,Meudon,19,20820.033285,39.153007,43.121913
986,92060,Le Plessis-Robinson,9,13241.207801,38.179758,28.479042
993,92075,Vanves,11,13506.150944,44.512056,50.622239
1018,93050,Neuilly-sur-Marne,15,15361.848596,33.936535,51.511378
1057,94046,Maisons-Alfort,23,26642.601943,41.178662,51.603543
1119,95127,Cergy,27,28094.092826,37.197948,50.509895
1220,95500,Pontoise,10,13575.620469,41.066272,38.0914


In [24]:
#Récupérer les géométries

contours = gpd.read_file(FICHIER_GEO,layer=COUCHE_GEO)
contours = verifier_cle(contours,'IRIS',9,'GeoPackage J13a')
if set(contours['IRIS'])!=ATTENDUS or contours.crs is None:
    raise ValueError('GeoPackage J13a : clés ou système de coordonnées incompatibles.')
if not (contours.geometry.notna() & ~contours.geometry.is_empty & contours.geometry.is_valid
        & contours.geometry.geom_type.isin(['Polygon','MultiPolygon'])).all():
    raise ValueError('Géométries J13a invalides.')
iris_geo = contours[['IRIS','geometry']].to_crs(2154).merge(profil,on='IRIS',how='left',validate='one_to_one')
iris_geo = gpd.GeoDataFrame(iris_geo,geometry='geometry',crs=2154)
affichage = iris_geo[['IRIS','geometry']].copy()
simple = affichage.geometry.simplify(SIMPLIFICATION_AFFICHAGE_METRES,preserve_topology=True)
valides = simple.is_valid & ~simple.is_empty
affichage.loc[valides,'geometry'] = simple.loc[valides]
affichage = affichage.to_crs(4326)
geojson = json.loads(affichage.to_json())
xmin,ymin,xmax,ymax = affichage.total_bounds
marge_lon = max((xmax-xmin)*.06,.002); marge_lat = max((ymax-ymin)*.06,.002)
FICHIERS_HTML = []




In [25]:
#Explorer les cartes Plotly

cartes = [
 ('J13B_PART_MENAGES_UNE_PERSONNE_PCT','Ménages d’une personne','% des ménages','01_menages_une_personne.html'),
 ('J13B_PART_SCOLARISES_1824_PCT','Scolarisation des 18–24 ans','% des 18–24 ans','02_scolarisation_18_24.html'),
 ('J13B_PART_ACTIFS_TRANSPORTS_COMMUN_PCT','Transports en commun pour aller travailler','% des actifs occupés 15+','03_transports_communs_residents.html'),
]
if iris_geo[COL_MED].notna().any():
    cartes.append((COL_MED,'Niveau de vie médian annuel par UC','€/UC/an','04_niveau_vie_median.html'))
def texte_nombre(v):
    return 'Non disponible' if pd.isna(v) else f'{float(v):,.1f}'.replace(',',' ').replace('.',',')
for col,titre,unite,nom in cartes:
    textes = ['<b>'+escape(str(r['NOM_IRIS']))+'</b><br>'+escape(str(r['NOM_COMMUNE_ETUDE']))
        +'<br>IRIS : '+escape(str(r['IRIS']))+'<br>'+escape(titre)+' : '+texte_nombre(r[col])
        +'<br>Unité : '+escape(unite)
        +('<br>Statut fiscal : '+escape(str(r['J13B_STATUT_REVENUS'])) if col==COL_MED else '')
        for _,r in iris_geo.iterrows()]
    fig = go.Figure(go.Choropleth(geojson=geojson,featureidkey='properties.IRIS',
        locations=iris_geo['IRIS'].tolist(),z=[0]*len(iris_geo),showscale=False,
        colorscale=[[0,'#dedede'],[1,'#dedede']],marker_line_color='white',marker_line_width=.4,
        text=textes,hovertemplate='%{text}<extra></extra>'))
    connus = iris_geo[col].notna()
    if connus.any():
        fig.add_trace(go.Choropleth(geojson=geojson,featureidkey='properties.IRIS',
            locations=iris_geo.loc[connus,'IRIS'].tolist(),z=iris_geo.loc[connus,col].astype(float).tolist(),
            colorscale='YlOrRd',marker_line_color='white',marker_line_width=.4,
            text=[textes[i] for i,b in enumerate(connus) if b],hovertemplate='%{text}<extra></extra>',
            colorbar_title=unite))
    fig.update_geos(visible=False,projection_type='mercator',
        lonaxis_range=[xmin-marge_lon,xmax+marge_lon],lataxis_range=[ymin-marge_lat,ymax+marge_lat])
    periode = f'Filosofi {ANNEE_FILOSOFI}, géographie source {GEOGRAPHIE_FILOSOFI}' if col==COL_MED else f'RP {ANNEE_RP}'
    fig.update_layout(title=titre+' — IDF · '+periode,height=750,margin=dict(l=10,r=10,t=70,b=10))
    fichier = DOSSIER_CARTES/nom
    fig.write_html(str(fichier),include_plotlyjs=True,full_html=True,config=CONFIG_PLOTLY)
    FICHIERS_HTML.append(fichier)
    if AFFICHER_CARTES_DANS_NOTEBOOK:
        fig.show(config=CONFIG_PLOTLY)
    print('Carte enregistrée :',fichier)



Carte enregistrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j13b\01_menages_une_personne.html
Carte enregistrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j13b\02_scolarisation_18_24.html
Carte enregistrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j13b\03_transports_communs_residents.html
Carte enregistrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j13b\04_niveau_vie_median.html


In [26]:
#Exporter les profils, focus

FICHIER_PROFIL = DOSSIER_PROCESSED/'profil_iris_idf_j13b.csv'
FICHIER_GEO_SORTIE = DOSSIER_PROCESSED/'profil_iris_idf_j13b.gpkg'
FICHIER_FOCUS = DOSSIER_PROCESSED/'profil_iris_focus_j13b.csv'
FICHIER_SYNTHESE = DOSSIER_PROCESSED/'synthese_indicateurs_rp_communes_idf_j13b.csv'
FICHIER_DICTIONNAIRE = DOSSIER_PROCESSED/'dictionnaire_indicateurs_j13b.csv'
profil_focus = profil[profil['J13B_DANS_FOCUS']].copy()
exporter_csv(profil,FICHIER_PROFIL)
exporter_csv(profil_focus,FICHIER_FOCUS)
exporter_csv(synthese,FICHIER_SYNTHESE)
table_dictionnaire = valider_dictionnaire()
exporter_csv(table_dictionnaire,FICHIER_DICTIONNAIRE)
if FICHIER_GEO_SORTIE.exists():
    FICHIER_GEO_SORTIE.unlink()
iris_geo.to_file(FICHIER_GEO_SORTIE,layer='profils_iris',driver='GPKG',index=False)
couverture = profil.groupby('DEPARTEMENT').agg(NB_IRIS=('IRIS','size'),
    NB_IRIS_FOCUS=('J13B_DANS_FOCUS','sum'),
    NB_IRIS_MEDIANE_REVENUS_DISPONIBLE=('J13B_MEDIANE_REVENUS_DISPONIBLE','sum')).reset_index()
exporter_csv(couverture,DOSSIER_INTERIM/'couverture_departements_j13b.csv')
display(couverture)
print('Profil régional :',FICHIER_PROFIL)
print('Focus local :',FICHIER_FOCUS)
print('Dictionnaire :',FICHIER_DICTIONNAIRE)


,DEPARTEMENT,NB_IRIS,NB_IRIS_FOCUS,NB_IRIS_MEDIANE_REVENUS_DISPONIBLE
0,75,992,0,864
1,77,761,0,273
2,78,683,13,449
3,91,507,0,322
4,92,616,80,584
5,93,614,15,568
6,94,530,23,513
7,95,561,37,380


Profil régional : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13b\profil_iris_idf_j13b.csv
Focus local : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13b\profil_iris_focus_j13b.csv
Dictionnaire : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13b\dictionnaire_indicateurs_j13b.csv


In [27]:
#Controles finaux et provenances

assert profil['IRIS'].is_unique and len(profil)==NOMBRE_IRIS_INITIAL
assert set(profil['IRIS'])==ATTENDUS
assert set(profil['DEPARTEMENT'])==DEPARTEMENTS_IDF
assert len(iris_geo)==len(profil) and iris_geo.geometry.is_valid.all()
assert int(profil['J13A_COMMUNE_PRESELECTIONNEE_J12'].sum())==int(base['J13A_COMMUNE_PRESELECTIONNEE_J12'].sum())
assert set(profil_focus['CODGEO_ETUDE'])==codes_focus
assert all(profil['J13B_LIGNE_'+t+'_PRESENTE'].all() for t in THEMES)
assert all(profil[c].dropna().ge(0).all() for c in COLONNES_EFFECTIFS)
assert all(p.is_file() for p in [FICHIER_PROFIL,FICHIER_GEO_SORTIE,FICHIER_FOCUS,FICHIER_SYNTHESE,FICHIER_DICTIONNAIRE])
assert len(FICHIERS_HTML)>=3 and all(p.is_file() for p in FICHIERS_HTML)
table_dictionnaire = valider_dictionnaire()
assert table_dictionnaire['COLONNE'].is_unique
exporter_csv(table_dictionnaire,FICHIER_DICTIONNAIRE)
sources = [FICHIER_BASE,FICHIER_GEO]+[f for f,_ in THEMES.values()]
if CHARGER_REVENUS:
    sources.append(FICHIER_REVENUS)
if FICHIER_CORRESPONDANCE_REVENUS is not None and CHARGER_REVENUS:
    sources.append(Path(FICHIER_CORRESPONDANCE_REVENUS))
metadata = {'etape':'J13B_PROFILS_IRIS_IDF','execution_utc':datetime.now(timezone.utc).isoformat(),
    'annee_rp':ANNEE_RP,'geographie_rp':GEOGRAPHIE_RP,'annee_filosofi':ANNEE_FILOSOFI,
    'geographie_filosofi':GEOGRAPHIE_FILOSOFI,'revenus_charges':CHARGER_REVENUS,
    'nb_iris_idf':len(profil),'nb_iris_focus':len(profil_focus),'codes_communes_focus':sorted(codes_focus),
    'nb_correspondances_revenus_validees':len(passage),'statuts_revenus':statuts_revenus.to_dict(orient='records'),
    'tolerance_absolue':TOLERANCE_ABSOLUE,'tolerance_relative':TOLERANCE_RELATIVE,
    'sources':[{'fichier':str(p),'sha256':sha256(p)} for p in dict.fromkeys(sources)],
    'mode_colonnes_manquantes':MODE_COLONNES_MANQUANTES,
    'schema_pcs_activite':SCHEMA_PCS_CHOISI,'statut_pcs_source':STATUT_PCS_SOURCE,
    'nb_colonnes_source_absentes':len(absentes_globales),
    'statut_execution':'TERMINE_AVEC_COLONNES_SOURCE_ABSENTES' if len(absentes_globales) else 'TERMINE_COLONNES_SOURCE_COMPLETES',
    'manifest_sources':MANIFESTE_SOURCES,'disponibilite_colonnes':DIAGNOSTIC_COLONNES,
    'couverture_indicateurs':couverture_indicateurs.to_dict(orient='records'),
    'variables_pcs_sources':({d:s for s,(d,_) in SCHEMAS_PCS[SCHEMA_PCS_CHOISI].items()
        if s in COLONNES_PRESENTES['ACTIVITE']} if SCHEMA_PCS_CHOISI is not None else {}),
    'lectures':LECTURES,'controles_jointures':CONTROLES_JOINTURES,
    'cartes':[str(p) for p in FICHIERS_HTML],'affichage_notebook':AFFICHER_CARTES_DANS_NOTEBOOK,
    'logiciels':{'pandas':pd.__version__,'geopandas':gpd.__version__,'plotly':plotly.__version__},
    'limites':['Caractéristiques des résidents, pas emplois ou étudiants présents sur place.',
       'Parts RP calculées sur des dénominateurs cohérents, NA conservés, aucun arrondi avant calcul.',
       'Revenus IRIS non sommés et non moyennés pour produire une médiane communale.',
       'Revenus d’une autre géographie joints seulement sur une correspondance un-à-un validée.',
       'Un logement vacant n’est pas un local commercial disponible.',
       'Les géographies des sources sont déclarées à partir des notices.']}
(DOSSIER_PROCESSED/'metadata_profils_iris_j13b.json').write_text(
    json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('J13b terminé : profils, focus, diagnostics, dictionnaire et cartes exportés.')
print('PCS :',STATUT_PCS_SOURCE,'; colonnes sources absentes :',len(absentes_globales))
if len(absentes_globales):
    print('Certains indicateurs restent indisponibles. Consulter colonnes_absentes_j13b.csv et couverture_indicateurs_j13b.csv.')


J13b terminé : profils, focus, diagnostics, dictionnaire et cartes exportés.
PCS : COMPLETE ; colonnes sources absentes : 0
